# LeetCode 4109 — Count Valid Suffix Sorts II (Hard)

Notebook này giảng **từ căn bản** bài toán:

```python
class Solution:
    def countValidSuffixSorts(self, nums: List[int], target: List[int]) -> List[int]:
```

> Ghi chú: trang đề mình dùng (bản mirror công khai leetcode.doocs.org) không in sẵn skeleton code.
> Tên method `countValidSuffixSorts` được suy ra theo quy ước đặt tên quen thuộc của LeetCode.
> Nếu trên LeetCode tên method khác, bạn chỉ cần đổi tên — phần thân giữ nguyên.

Đây là bài Hard (Weekly Contest 527 — câu 4, rating ≈ 2384). Thoạt nhìn đề có vẻ bắt ta
"sắp xếp rất nhiều lần rồi so sánh rất nhiều lần". Nhưng sau vài quan sát, mọi thứ thu về
**3 câu hỏi nhỏ cho mỗi cửa sổ** của `target`, và mỗi câu hỏi đều có một kỹ thuật kinh điển trả lời nhanh:

- **Sliding window đếm tần suất** — cửa sổ có phải là một hoán vị của `nums` không?
- **Mảng tiền xử lý vị trí "đi lên"** — đoạn cuối không tăng của cửa sổ bắt đầu từ đâu?
- **LCP bằng rolling hash hoặc Z-function** — cửa sổ trùng với `nums` được bao nhiêu phần tử đầu?

---

## Sau notebook này, bạn sẽ nắm được

1. Đề bài thật sự yêu cầu gì: mảng "sau khi sắp xếp hậu tố" trông như thế nào, và vì sao một cửa sổ có thể được tạo ra bởi **nhiều** chỉ số `i`.
2. Cách viết brute force đúng tuyệt đối theo đề, và **đếm** được nó chậm đến mức nào.
3. Cách gom các mảng giống nhau bằng `Counter` với key là `tuple`.
4. **Bước đột phá**: tách điều kiện "hai mảng bằng nhau" thành 3 điều kiện độc lập, rồi nhận ra tập các `i` hợp lệ là **một đoạn liên tục**.
5. Kỹ thuật **sliding window** kiểm tra "cửa sổ là hoán vị của `nums`" với chi phí O(1) mỗi lần trượt.
6. Kỹ thuật **polynomial rolling hash** kết hợp **binary search** để tính độ dài tiền tố chung (LCP).
7. Kỹ thuật **Z-function** tính LCP của `nums` với mọi vị trí trong `target` trong O(n + m), không có rủi ro va chạm hash.

---

## Các cách giải sẽ học (từ dễ đến tối ưu)

| Cách | Ý tưởng chính | Thời gian |
|---|---|---|
| Cách 1 | Brute force: với mỗi cửa sổ, dựng lại từng mảng đã sắp xếp rồi so sánh | O((m−n+1)·n²·log n) |
| Cách 2 | Dựng `n` mảng **một lần**, gom bằng `Counter` (key là `tuple`), tra từng cửa sổ | O(n²·log n + (m−n+1)·n) |
| Cách 3 | 3 điều kiện + sliding window + mảng "đi lên" + LCP bằng **rolling hash & binary search** | O(n + m·log n) |
| Cách 4 | Như Cách 3 nhưng LCP bằng **Z-function** | O(n + m) |

# 1. Đề bài

## Phát biểu (dịch sát đề gốc)

Cho hai mảng số nguyên `nums` và `target` có độ dài lần lượt là `n` và `m`, với `n <= m`.

Với **mỗi** chỉ số `i`, hãy xét việc **sắp xếp hậu tố** `nums[i..n - 1]` của **mảng ban đầu** theo thứ tự **không tăng** (non-increasing).

Trả về mảng số nguyên `ans` có độ dài `m - n + 1`, trong đó `ans[k]` là **số chỉ số `i`** sao cho mảng thu được **bằng** `target[k..k + n - 1]`.

**Hậu tố** (suffix) của một mảng là mảng con bắt đầu ở một chỉ số bất kỳ và kéo dài tới cuối mảng.

---

## Giải nghĩa từng cụm từ

- `nums[i..n - 1]`: đoạn từ chỉ số `i` đến `n - 1`, **lấy cả hai đầu**. Trong Python viết là `nums[i:]`.
- **Không tăng** (non-increasing): mỗi phần tử **lớn hơn hoặc bằng** phần tử đứng sau. Ví dụ `[3, 2, 2, 1]` là không tăng.
  Khác với **giảm ngặt** (strictly decreasing) như `[3, 2, 1]`, vì "không tăng" **cho phép bằng nhau**.
- **Mảng ban đầu**: mỗi chỉ số `i` làm việc trên bản gốc của `nums`; các lần sắp xếp **không cộng dồn**.
- Với chỉ số `i`, mảng thu được giữ nguyên `nums[0..i-1]` và thay `nums[i..n-1]` bằng bản đã sắp xếp không tăng.
  Trong notebook ta gọi mảng này là **`A_i`**. Có đúng `n` mảng: `A_0, A_1, ..., A_{n-1}`.
- `target[k..k + n - 1]` là **cửa sổ** (window) độ dài `n` bắt đầu tại `k`. Có `m - n + 1` cửa sổ, `k = 0, 1, ..., m - n`. Ta gọi cửa sổ thứ `k` là **`W_k`**.

Vậy câu hỏi gọn lại là:

> Với mỗi cửa sổ `W_k`, có **bao nhiêu chỉ số `i`** thỏa `A_i == W_k`?

---

## Ví dụ 1

```text
Input:  nums = [2,2,3], target = [2,2,3,2,1,3,2,2]
Output: [1,1,0,0,0,1]
```

Sắp xếp hậu tố bắt đầu tại `i = 0, 1, 2` cho ra `[3,2,2]`, `[2,3,2]`, `[2,2,3]`.

- `k = 0`: `[2,2,3]` được tạo bởi `i = 2`.
- `k = 1`: `[2,3,2]` được tạo bởi `i = 1`.
- `k = 2, 3, 4`: không cửa sổ nào được tạo ra.
- `k = 5`: `[3,2,2]` được tạo bởi `i = 0`.

## Ví dụ 2

```text
Input:  nums = [5,6,5], target = [5,6,5,5,6,5]
Output: [2,1,0,2]
```

Hậu tố từ `i = 0` cho `[6,5,5]`; từ `i = 1` **hoặc** `i = 2` đều cho `[5,6,5]`.

- `k = 0`: `[5,6,5]` được tạo bởi `i = 1` và `i = 2` → đáp án `2`.
- `k = 1`: `[6,5,5]` được tạo bởi `i = 0`.
- `k = 2`: `[5,5,6]` không được tạo ra.
- `k = 3`: `[5,6,5]` được tạo bởi `i = 1` và `i = 2`.

---

## Ràng buộc (constraints)

- `n == nums.length`, `m == target.length`
- `1 <= n <= m <= 10^5`
- `1 <= nums[i], target[i] <= 10^5`

## Ràng buộc nói gì về độ phức tạp?

Hãy đếm "khối lượng" tự nhiên của bài:

- Có `m - n + 1` cửa sổ và `n` chỉ số `i`, tức `(m - n + 1) · n` cặp `(k, i)`.
- Tích này lớn nhất khi `n ≈ m / 2`: khoảng `50 001 · 50 000 ≈ 2.5 · 10^9` cặp.
- Ngay cả việc **chỉ đọc hết** mỗi cửa sổ một lần cũng tốn `(m - n + 1) · n` lần đọc — cũng cỡ `2.5 · 10^9`.

Python thuần làm được khoảng `10^7` thao tác đơn giản mỗi giây. Vậy:

> Ta **không được** đọc lại toàn bộ cửa sổ cho mỗi `k`. Mỗi cửa sổ chỉ được tốn **O(1)** hoặc **O(log n)** công việc,
> tổng cộng khoảng **O(n + m)** hoặc **O((n + m) · log n)**.

Hai chi tiết nhỏ nhưng hữu ích:

- Giá trị nằm trong `[1, 10^5]` → có thể dùng **mảng đếm** kích thước `10^5 + 1` thay vì `dict`.
- Giá trị luôn `>= 1` → số `0` có thể dùng làm **ký tự phân cách** (separator) khi ghép mảng (sẽ cần ở Z-function).

In [1]:
from typing import List

# Dữ liệu của đề: (nums, target, expected)
examples = [
    ([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2], [1, 1, 0, 0, 0, 1]),
    ([5, 6, 5], [5, 6, 5, 5, 6, 5], [2, 1, 0, 2]),
]

for nums, target, expected in examples:
    n, m = len(nums), len(target)
    print(f"nums = {nums}  (n = {n})")
    print(f"target = {target}  (m = {m})")
    print(f"Số cửa sổ = m - n + 1 = {m - n + 1}")
    for k in range(m - n + 1):
        print(f"  W_{k} = target[{k}..{k + n - 1}] = {target[k:k + n]}")
    print("Kết quả mong đợi:", expected)
    print()

nums = [2, 2, 3]  (n = 3)
target = [2, 2, 3, 2, 1, 3, 2, 2]  (m = 8)
Số cửa sổ = m - n + 1 = 6
  W_0 = target[0..2] = [2, 2, 3]
  W_1 = target[1..3] = [2, 3, 2]
  W_2 = target[2..4] = [3, 2, 1]
  W_3 = target[3..5] = [2, 1, 3]
  W_4 = target[4..6] = [1, 3, 2]
  W_5 = target[5..7] = [3, 2, 2]
Kết quả mong đợi: [1, 1, 0, 0, 0, 1]

nums = [5, 6, 5]  (n = 3)
target = [5, 6, 5, 5, 6, 5]  (m = 6)
Số cửa sổ = m - n + 1 = 4
  W_0 = target[0..2] = [5, 6, 5]
  W_1 = target[1..3] = [6, 5, 5]
  W_2 = target[2..4] = [5, 5, 6]
  W_3 = target[3..5] = [5, 6, 5]
Kết quả mong đợi: [2, 1, 0, 2]



# 2. Hiểu đề bằng tay (chưa cần thuật toán)

## Bước 1 — Dựng các mảng `A_i` cho Ví dụ 1

`nums = [2, 2, 3]`

| i | giữ nguyên `nums[:i]` | hậu tố `nums[i:]` | hậu tố sau khi sắp xếp không tăng | `A_i` |
|---|---|---|---|---|
| 0 | `[]` | `[2, 2, 3]` | `[3, 2, 2]` | `[3, 2, 2]` |
| 1 | `[2]` | `[2, 3]` | `[3, 2]` | `[2, 3, 2]` |
| 2 | `[2, 2]` | `[3]` | `[3]` | `[2, 2, 3]` |

Để ý dòng cuối: với `i = n - 1`, hậu tố chỉ có **1 phần tử**, sắp xếp không làm gì cả, nên

> `A_{n-1}` **luôn bằng chính `nums`**.

## Bước 2 — So từng cửa sổ với các `A_i`

| k | `W_k` | bằng `A_i` nào? | `ans[k]` |
|---|---|---|---|
| 0 | `[2, 2, 3]` | `A_2` | 1 |
| 1 | `[2, 3, 2]` | `A_1` | 1 |
| 2 | `[3, 2, 1]` | không có (có số 1, `nums` không có) | 0 |
| 3 | `[2, 1, 3]` | không có | 0 |
| 4 | `[1, 3, 2]` | không có | 0 |
| 5 | `[3, 2, 2]` | `A_0` | 1 |

Kết quả `[1, 1, 0, 0, 0, 1]` — khớp đề.

In [2]:
nums = [2, 2, 3]
n = len(nums)

for i in range(n):
    giu_nguyen = nums[:i]                     # phần đầu, không đụng tới
    hau_to = nums[i:]                         # phần sẽ được sắp xếp
    hau_to_da_sap = sorted(hau_to, reverse=True)
    A_i = giu_nguyen + hau_to_da_sap
    print(f"i = {i}: {giu_nguyen} + sorted({hau_to}, giảm dần) = {giu_nguyen} + {hau_to_da_sap} = {A_i}")

i = 0: [] + sorted([2, 2, 3], giảm dần) = [] + [3, 2, 2] = [3, 2, 2]
i = 1: [2] + sorted([2, 3], giảm dần) = [2] + [3, 2] = [2, 3, 2]
i = 2: [2, 2] + sorted([3], giảm dần) = [2, 2] + [3] = [2, 2, 3]


## Bước 3 — Ví dụ 2: vì sao `ans[k]` có thể bằng 2?

`nums = [5, 6, 5]`:

- `A_0 = sorted([5, 6, 5], giảm dần) = [6, 5, 5]`
- `A_1 = [5] + sorted([6, 5], giảm dần) = [5, 6, 5]`
- `A_2 = [5, 6] + [5] = [5, 6, 5]`

`A_1` và `A_2` **giống hệt nhau** dù là hai chỉ số khác nhau. Đề đếm **số chỉ số `i`**, không phải số mảng khác nhau, nên cửa sổ `[5, 6, 5]` được tính **2**.

Cửa sổ `W_2 = [5, 5, 6]` thì sao? Nó có **đúng các phần tử** của `nums` (hai số 5, một số 6) — tức là một **hoán vị** (permutation) của `nums` — nhưng không `A_i` nào bằng nó.

## Hai bài học rút ra ngay từ ví dụ

1. **Đếm chỉ số, không đếm mảng khác nhau.** Nhiều `i` có thể cho cùng một mảng.
2. **"Là hoán vị của `nums`" là điều kiện cần nhưng chưa đủ.** Mỗi `A_i` còn có "hình dạng" đặc biệt mà `[5, 5, 6]` không có.

Bài học thứ 2 chính là hạt giống của bước đột phá ở mục 6.

In [3]:
nums = [5, 6, 5]
target = [5, 6, 5, 5, 6, 5]
n, m = len(nums), len(target)

A = [nums[:i] + sorted(nums[i:], reverse=True) for i in range(n)]
for i in range(n):
    print(f"A_{i} = {A[i]}")
print()

for k in range(m - n + 1):
    W = target[k:k + n]
    matched = [i for i in range(n) if A[i] == W]
    print(f"k = {k}: W = {W}, khớp các i = {matched} -> ans[{k}] = {len(matched)}")

A_0 = [6, 5, 5]
A_1 = [5, 6, 5]
A_2 = [5, 6, 5]

k = 0: W = [5, 6, 5], khớp các i = [1, 2] -> ans[0] = 2
k = 1: W = [6, 5, 5], khớp các i = [0] -> ans[1] = 1
k = 2: W = [5, 5, 6], khớp các i = [] -> ans[2] = 0
k = 3: W = [5, 6, 5], khớp các i = [1, 2] -> ans[3] = 2


# 3. Nền tảng (Python + thuật toán)

Trước khi giải, ta học các "viên gạch" sẽ dùng. Mỗi viên gạch: **là gì → ví dụ nhỏ → code chạy thử → liên hệ với bài**.

| Mục | Công cụ | Dùng ở đâu trong bài |
|---|---|---|
| 3.1 | Slicing `a[:i]`, `a[i:]`, nối list `+`, `sorted(..., reverse=True)` | Dựng `A_i` (Cách 1, 2) |
| 3.2 | `tuple` làm key, `Counter` | Gom các `A_i` giống nhau (Cách 2) |
| 3.3 | Sliding window đếm tần suất | Kiểm tra cửa sổ là hoán vị của `nums` (Cách 3, 4) |
| 3.4 | Dãy không tăng, vị trí "đi lên", mảng tiền xử lý | Tìm đoạn cuối không tăng của cửa sổ (Cách 3, 4) |
| 3.5 | Polynomial rolling hash, phép `%` (modulo) | So sánh hai đoạn mảng trong O(1) (Cách 3) |
| 3.6 | Binary search trên một điều kiện đơn điệu | Tính độ dài tiền tố chung LCP (Cách 3) |
| 3.7 | Z-function | Tính LCP cho mọi vị trí trong O(n + m) (Cách 4) |

## 3.1 Slicing, nối list và `sorted(..., reverse=True)`

**Slicing** (cắt lát) lấy một đoạn con của list:

```python
a = [2, 2, 3]
a[:1]   # từ đầu đến TRƯỚC chỉ số 1  -> [2]
a[1:]   # từ chỉ số 1 đến hết        -> [2, 3]
```

Đọc `a[:i]` là "`i` phần tử đầu", còn `a[i:]` là "từ vị trí `i` đến hết" — chính là **hậu tố** bắt đầu tại `i`.

**Nối list** bằng `+` tạo ra **list mới**: `[2] + [3, 2]` → `[2, 3, 2]`.

**`sorted(x, reverse=True)`** trả về một **list mới** đã sắp xếp **giảm dần (không tăng)**; list cũ không bị thay đổi.
Với phần tử trùng nhau, kết quả vẫn là không tăng: `sorted([5, 6, 5], reverse=True)` → `[6, 5, 5]`.

Liên hệ với bài: `A_i = nums[:i] + sorted(nums[i:], reverse=True)`.

In [4]:
a = [5, 6, 5]
print("a[:1]  =", a[:1])
print("a[1:]  =", a[1:])
print("sorted(a, reverse=True) =", sorted(a, reverse=True))
print("a sau khi gọi sorted vẫn là", a, "(sorted không sửa list gốc)")

def sorted_suffix_array(nums, i):
    """Mảng A_i: giữ nguyên nums[:i], hậu tố nums[i:] được sắp xếp không tăng."""
    return nums[:i] + sorted(nums[i:], reverse=True)

for i in range(len(a)):
    print(f"A_{i} =", sorted_suffix_array(a, i))

a[:1]  = [5]
a[1:]  = [6, 5]
sorted(a, reverse=True) = [6, 5, 5]
a sau khi gọi sorted vẫn là [5, 6, 5] (sorted không sửa list gốc)
A_0 = [6, 5, 5]
A_1 = [5, 6, 5]
A_2 = [5, 6, 5]


## 3.2 `tuple` làm key của `dict`, và `Counter`

`dict` chỉ nhận key **hashable** (băm được — Python tính được một "mã số" cố định cho nó).
`list` có thể bị sửa nên **không** hashable; `tuple` thì **không sửa được** nên hashable.

Vì vậy muốn đếm "mảng nào xuất hiện bao nhiêu lần", ta đổi mảng sang `tuple` trước.

`Counter` (trong `collections`) là một `dict` chuyên đếm: `Counter(iterable)` đếm số lần mỗi phần tử xuất hiện, và đọc key không có thì trả về `0` thay vì báo lỗi.

Thêm một khái niệm quan trọng: **multiset** (đa tập hợp) — "tập hợp cho phép lặp lại".
Hai mảng có **cùng multiset** ⇔ mảng này là **hoán vị** của mảng kia ⇔ `Counter(a) == Counter(b)`.

Liên hệ với bài: Cách 2 dùng `Counter` các `tuple(A_i)`; Bước đột phá dùng ý "cùng multiset".

In [5]:
from collections import Counter

d = {}
try:
    d[[1, 2]] = 1               # list làm key -> lỗi
except TypeError as e:
    print("Lỗi:", e)

d[(1, 2)] = 1                   # tuple làm key -> OK
print("d =", d)

A = [[6, 5, 5], [5, 6, 5], [5, 6, 5]]          # A_0, A_1, A_2 của Ví dụ 2
freq = Counter(tuple(a) for a in A)
print("freq =", freq)
print("freq[(5, 6, 5)] =", freq[(5, 6, 5)])
print("freq[(5, 5, 6)] =", freq[(5, 5, 6)], "(không có -> 0)")

print("Cùng multiset?", Counter([5, 5, 6]) == Counter([5, 6, 5]))
print("Cùng multiset?", Counter([5, 5, 6]) == Counter([5, 6, 6]))

Lỗi: unhashable type: 'list'
d = {(1, 2): 1}
freq = Counter({(5, 6, 5): 2, (6, 5, 5): 1})
freq[(5, 6, 5)] = 2
freq[(5, 5, 6)] = 0 (không có -> 0)
Cùng multiset? True
Cùng multiset? False


## 3.3 Sliding window đếm tần suất — "cửa sổ có phải hoán vị không?"

**Bài toán con:** với mỗi cửa sổ độ dài `n` của `text`, cửa sổ đó có cùng multiset với `pattern` không?

Cách ngây thơ: mỗi cửa sổ gọi `Counter(...)` → O(n) mỗi cửa sổ → quá chậm.

**Sliding window** (cửa sổ trượt): khi trượt từ cửa sổ `k - 1` sang `k`, chỉ có **2 phần tử thay đổi**:

```text
cửa sổ k-1:  [ text[k-1], text[k], ..., text[k+n-2] ]
cửa sổ k:               [ text[k], ..., text[k+n-2], text[k+n-1] ]
               ^ rời cửa sổ                          ^ vào cửa sổ
```

Ta duy trì:

- `diff[v] = (số lần v trong pattern) - (số lần v trong cửa sổ)`
- `bad = số giá trị v có diff[v] != 0` (số giá trị đang "lệch")

Cửa sổ là hoán vị của `pattern` ⇔ **`bad == 0`**.

Khi một `diff[v]` thay đổi:

- từ `0` thành khác `0` → `bad += 1` (vừa bị lệch),
- từ khác `0` thành `0` → `bad -= 1` (vừa cân bằng lại),
- còn lại → `bad` giữ nguyên.

Hai công cụ Python mới trong code dưới:

- `defaultdict(int)`: dict mà key chưa có sẽ tự nhận giá trị `0`.
- `nonlocal bad`: cho phép hàm con `change` **sửa** biến `bad` của hàm bao ngoài (không có dòng này, Python sẽ coi `bad` là biến cục bộ mới và báo lỗi).

In [6]:
from collections import defaultdict

def demo_permutation_windows(pattern, text):
    n = len(pattern)
    diff = defaultdict(int)   # diff[v] = #v trong pattern - #v trong cửa sổ
    bad = 0                   # số giá trị v có diff[v] != 0

    def change(v, d):
        nonlocal bad
        before = diff[v]
        diff[v] += d
        after = diff[v]
        if before == 0 and after != 0:
            bad += 1
        elif before != 0 and after == 0:
            bad -= 1

    for x in pattern:              # pattern góp dấu +
        change(x, +1)
    for j in range(n):             # cửa sổ đầu tiên góp dấu -
        change(text[j], -1)

    for k in range(len(text) - n + 1):
        if k > 0:
            change(text[k - 1], +1)      # phần tử rời cửa sổ: trả lại
            change(text[k + n - 1], -1)  # phần tử vào cửa sổ: trừ đi
        lech = {v: d for v, d in diff.items() if d != 0}
        print(f"k={k}: cửa sổ={text[k:k + n]}, bad={bad}, đang lệch={lech}, hoán vị? {bad == 0}")

demo_permutation_windows([1, 1, 2], [1, 2, 1, 3, 1, 1, 2])

k=0: cửa sổ=[1, 2, 1], bad=0, đang lệch={}, hoán vị? True
k=1: cửa sổ=[2, 1, 3], bad=2, đang lệch={1: 1, 3: -1}, hoán vị? False
k=2: cửa sổ=[1, 3, 1], bad=2, đang lệch={2: 1, 3: -1}, hoán vị? False
k=3: cửa sổ=[3, 1, 1], bad=2, đang lệch={2: 1, 3: -1}, hoán vị? False
k=4: cửa sổ=[1, 1, 2], bad=0, đang lệch={}, hoán vị? True


## 3.4 Dãy không tăng và vị trí "đi lên"

Gọi `t` là một **vị trí đi lên** (ascent) của mảng `a` nếu `a[t] < a[t + 1]`.

- Một đoạn là **không tăng** ⇔ bên trong đoạn **không có** vị trí đi lên nào.
- **Hậu tố không tăng dài nhất** của `a` bắt đầu ngay sau vị trí đi lên **cuối cùng**.
  Nếu không có vị trí đi lên nào, cả mảng là không tăng, hậu tố bắt đầu tại `0`.

Ví dụ `a = [5, 3, 3, 4, 2, 2, 1]`:

- Các cặp kề nhau: `5≥3`, `3≥3`, `3<4` ← đi lên tại `t = 2`, `4≥2`, `2≥2`, `2≥1`.
- Vị trí đi lên cuối cùng là `t = 2` → hậu tố không tăng dài nhất bắt đầu tại `s = 3`: `[4, 2, 2, 1]`.

**Mảng tiền xử lý `last_up`**: `last_up[t]` = vị trí đi lên **lớn nhất mà `<= t`** (hoặc `-1` nếu chưa có).
Tính một lần từ trái sang phải trong O(m). Sau đó, với **bất kỳ đoạn nào** `a[k..r]`, vị trí đi lên cuối cùng **bên trong đoạn** là `last_up[r - 1]` (nếu nó `>= k`).

> Tại sao lại là `r - 1`? Vì cặp cuối cùng nằm trọn trong đoạn là `(r - 1, r)`. Cặp `(r, r + 1)` đã thò ra ngoài đoạn.

Liên hệ với bài: phần sau `i` của mỗi `A_i` là **không tăng**, nên ta sẽ cần "hậu tố không tăng dài nhất của mỗi cửa sổ" thật nhanh.

In [7]:
a = [5, 3, 3, 4, 2, 2, 1]
ups = [t for t in range(len(a) - 1) if a[t] < a[t + 1]]
print("Các vị trí đi lên:", ups)

# Cách trực tiếp: đi lùi từ cuối khi còn a[s-1] >= a[s]
s = len(a) - 1
while s > 0 and a[s - 1] >= a[s]:
    s -= 1
print("Hậu tố không tăng dài nhất bắt đầu tại s =", s, "->", a[s:])

# Mảng tiền xử lý last_up
last_up = [-1] * len(a)
prev = -1
for t in range(len(a) - 1):
    if a[t] < a[t + 1]:
        prev = t
    last_up[t] = prev
print("last_up =", last_up, "(phần tử cuối không dùng tới)")

# Dùng last_up cho đoạn a[k..r]: vị trí đi lên cuối cùng bên trong là last_up[r-1]
for k, r in [(0, 6), (3, 6), (1, 3)]:
    j = last_up[r - 1]
    start = j + 1 if j >= k else k
    print(f"đoạn a[{k}..{r}] = {a[k:r + 1]}: hậu tố không tăng dài nhất bắt đầu tại {start} -> {a[start:r + 1]}")

Các vị trí đi lên: [2]
Hậu tố không tăng dài nhất bắt đầu tại s = 3 -> [4, 2, 2, 1]
last_up = [-1, -1, 2, 2, 2, 2, -1] (phần tử cuối không dùng tới)
đoạn a[0..6] = [5, 3, 3, 4, 2, 2, 1]: hậu tố không tăng dài nhất bắt đầu tại 3 -> [4, 2, 2, 1]
đoạn a[3..6] = [4, 2, 2, 1]: hậu tố không tăng dài nhất bắt đầu tại 3 -> [4, 2, 2, 1]
đoạn a[1..3] = [3, 3, 4]: hậu tố không tăng dài nhất bắt đầu tại 3 -> [4]


## 3.5 Polynomial rolling hash — so sánh hai đoạn mảng trong O(1)

**Ý tưởng:** biến mỗi đoạn mảng thành **một con số**, giống như viết các phần tử thành "chữ số" trong hệ cơ số `BASE`.

Với `BASE = 10` (chỉ để minh họa, không có modulo): `[2, 2, 3]` → `2·10² + 2·10¹ + 3·10⁰ = 223`.
Hai đoạn bằng nhau thì "con số" bằng nhau. Nhưng số sẽ rất lớn, nên trong thực tế ta lấy **phần dư** khi chia cho một số nguyên tố lớn `MOD` (phép `%`, gọi là **modulo**).

**Prefix hash** (hash tiền tố): `h[0] = 0`, `h[i + 1] = (h[i] · BASE + a[i]) % MOD`. Khi đó `h[r]` là hash của `a[0:r]`.

**Hash của đoạn `a[l:r]`** (lấy `l`, không lấy `r`):

```text
hash(a[l:r]) = (h[r] - h[l] · BASE^(r - l)) % MOD
```

Kiểm tra bằng số thập phân với `a = [2, 2, 3, 2]`: `h[4] = 2232`, `h[1] = 2`, nên
`hash(a[1:4]) = 2232 - 2 · 10³ = 232` — đúng là "232" ứng với `[2, 3, 2]`.

**Chọn tham số an toàn:**

- `BASE` phải **lớn hơn mọi giá trị** (ở đây `> 10^5`), nếu không các "chữ số" chồng lên nhau và hai mảng khác nhau dễ ra cùng hash (xem Lỗi 6 ở mục 14).
- `MOD` rất lớn, ở đây `2^61 - 1` (một số nguyên tố). Python có số nguyên lớn tùy ý nên nhân hai số ~`2^61` vẫn chính xác.
- Hash **có thể va chạm** (hai đoạn khác nhau cùng hash), nhưng với `MOD ≈ 2.3·10^18` xác suất cực nhỏ. Đây là thuật toán **xác suất** (probabilistic).

Liên hệ với bài: so sánh "`k`-phần tử đầu của `nums`" với "đoạn `target[k:k+L]`" trong O(1).

In [8]:
MOD = (1 << 61) - 1      # 2^61 - 1, một số nguyên tố rất lớn
BASE = 1_000_003         # lớn hơn mọi giá trị (<= 10^5)

def prefix_hash(a):
    h = [0] * (len(a) + 1)
    for i, x in enumerate(a):
        h[i + 1] = (h[i] * BASE + x) % MOD
    return h

def build_powers(n):
    pw = [1] * (n + 1)
    for i in range(n):
        pw[i + 1] = pw[i] * BASE % MOD
    return pw

def get_hash(h, pw, l, r):
    """Hash của đoạn a[l:r] (lấy l, không lấy r)."""
    return (h[r] - h[l] * pw[r - l]) % MOD

# Minh họa trực giác với BASE = 10, không modulo
a = [2, 2, 3, 2]
h10 = [0]
for x in a:
    h10.append(h10[-1] * 10 + x)
print("prefix 'thập phân' h10 =", h10)
print("đoạn a[1:4] ->", h10[4] - h10[1] * 10 ** 3)

# Hash thật
t = [2, 2, 3, 2, 1, 3, 2, 2]
h = prefix_hash(t)
pw = build_powers(len(t))
print("hash(t[0:2]) == hash(t[6:8]) ?", get_hash(h, pw, 0, 2) == get_hash(h, pw, 6, 8), " (", t[0:2], "vs", t[6:8], ")")
print("hash(t[0:3]) == hash(t[5:8]) ?", get_hash(h, pw, 0, 3) == get_hash(h, pw, 5, 8), " (", t[0:3], "vs", t[5:8], ")")

prefix 'thập phân' h10 = [0, 2, 22, 223, 2232]
đoạn a[1:4] -> 232
hash(t[0:2]) == hash(t[6:8]) ? True  ( [2, 2] vs [2, 2] )
hash(t[0:3]) == hash(t[5:8]) ? False  ( [2, 2, 3] vs [3, 2, 2] )


## 3.6 LCP và binary search trên một điều kiện đơn điệu

**LCP** (Longest Common Prefix — tiền tố chung dài nhất) của hai mảng `a`, `b` là số `L` lớn nhất sao cho `a[:L] == b[:L]`.

Xét điều kiện `ok(l) = (a[:l] == b[:l])`. Nếu `ok(l)` đúng thì `ok(l - 1)` cũng đúng (bỏ bớt phần tử cuối vẫn bằng nhau). Vậy dãy giá trị có dạng:

```text
l    : 0     1     2     3     4      5      6
ok(l): True  True  True  True  False  False  False
```

Một điều kiện "đúng rồi sai mãi" như vậy gọi là **đơn điệu** (monotone). Ta tìm **chỗ chuyển** bằng **binary search** (tìm kiếm nhị phân): mỗi bước loại một nửa khoảng → O(log n) lần kiểm tra.

Chi tiết dễ sai: ta tìm `l` **lớn nhất** mà `ok(l)` đúng, nên khi `ok(mid)` đúng thì gán `lo = mid`.
Phải lấy `mid = (lo + hi + 1) // 2` (làm tròn **lên**); nếu làm tròn xuống, khi `hi = lo + 1` ta có `mid = lo` và vòng lặp chạy mãi.

Liên hệ với bài: kết hợp với hash ở 3.5, mỗi lần kiểm tra `ok(mid)` chỉ tốn O(1) → LCP tốn O(log n).

In [9]:
def lcp_binary_search_demo(a, b):
    lo, hi = 0, min(len(a), len(b))      # đáp án nằm trong [lo, hi]
    while lo < hi:
        mid = (lo + hi + 1) // 2          # làm tròn lên
        ok = a[:mid] == b[:mid]           # ở đây so sánh trực tiếp cho dễ hiểu
        print(f"  lo={lo}, hi={hi}, mid={mid}, a[:{mid}] == b[:{mid}]? {ok}")
        if ok:
            lo = mid
        else:
            hi = mid - 1
    return lo

print("LCP =", lcp_binary_search_demo([5, 6, 5, 5, 6, 5], [5, 6, 5, 6, 6, 5]))

  lo=0, hi=6, mid=3, a[:3] == b[:3]? True
  lo=3, hi=6, mid=5, a[:5] == b[:5]? False
  lo=3, hi=4, mid=4, a[:4] == b[:4]? False
LCP = 3


## 3.7 Z-function — LCP với mọi vị trí trong O(N)

**Định nghĩa.** Với mảng `s` độ dài `N`, `z[i]` = độ dài tiền tố chung dài nhất của `s` và `s[i:]`.
(Quy ước ở notebook này: `z[0] = N`.)

Ví dụ `s = [1, 1, 2, 1, 1, 2, 1]`: `z[3] = 4` vì `s[3:] = [1, 1, 2, 1]` trùng 4 phần tử đầu của `s`.

**Cách ngây thơ:** với mỗi `i`, so từng phần tử → O(N²) trong trường hợp xấu (ví dụ mảng toàn số giống nhau).

**Mẹo "hộp" `[l, r)`:** luôn nhớ đoạn khớp **xa nhất về bên phải** đã tìm được: `s[l:r] == s[0:r - l]`.

- Nếu `i < r` thì `i` nằm trong hộp. Vì `s[l:r]` là bản sao của `s[0:r-l]`, phần `s[i:r]` là bản sao của `s[i-l : r-l]`.
  Do đó ta **đã biết trước** `z[i] >= min(r - i, z[i - l])` mà không cần so sánh lại.
- Sau đó mới so sánh thêm từng phần tử để kéo dài `z[i]` (chỉ cần khi chạm tới mép `r`).
- Nếu `i + z[i] > r`, cập nhật hộp mới `l, r = i, i + z[i]`.

**Vì sao O(N)?** Mỗi lần so sánh thành công trong vòng `while` đều đẩy `r` sang phải ít nhất 1 (hoặc xảy ra ở lần đầu đi qua vùng mới),
mà `r` chỉ tăng tối đa tới `N`. Tổng số so sánh thành công ≤ `N`, số so sánh thất bại ≤ 1 cho mỗi `i`.

In [10]:
def z_naive(s):
    N = len(s)
    z = [0] * N
    for i in range(N):
        while i + z[i] < N and s[z[i]] == s[i + z[i]]:
            z[i] += 1
    return z

def z_function(s):
    N = len(s)
    z = [0] * N
    if N == 0:
        return z
    z[0] = N
    l, r = 0, 0                      # hộp [l, r): s[l:r] == s[0:r-l]
    for i in range(1, N):
        if i < r:                    # i nằm trong hộp: dùng lại kết quả cũ
            z[i] = min(r - i, z[i - l])
        while i + z[i] < N and s[z[i]] == s[i + z[i]]:
            z[i] += 1                # kéo dài bằng so sánh trực tiếp
        if i + z[i] > r:             # tìm được hộp xa hơn
            l, r = i, i + z[i]
    return z

s = [1, 1, 2, 1, 1, 2, 1]
print("s       =", s)
print("z_naive =", z_naive(s))
print("z_func  =", z_function(s))

import random
rng = random.Random(0)
for _ in range(300):
    arr = [rng.randint(1, 3) for _ in range(rng.randint(1, 12))]
    assert z_naive(arr) == z_function(arr)
print("z_function khớp z_naive trên 300 mảng ngẫu nhiên")

s       = [1, 1, 2, 1, 1, 2, 1]
z_naive = [7, 1, 0, 4, 1, 0, 1]
z_func  = [7, 1, 0, 4, 1, 0, 1]
z_function khớp z_naive trên 300 mảng ngẫu nhiên


### Mẹo ghép `pattern + [phân cách] + text`

Muốn biết LCP của `pattern` với **mọi** hậu tố `text[k:]`, ta ghép:

```text
s = pattern + [0] + text
```

rồi chạy Z-function một lần. Khi đó `z[len(pattern) + 1 + k]` chính là LCP của `pattern` và `text[k:]`.

Vai trò của số `0` (separator): vì mọi giá trị đều `>= 1`, số `0` không bao giờ khớp với phần tử nào,
nên phép so khớp bắt đầu trong `text` **không thể vượt quá** độ dài `pattern` → `z` tự động `<= len(pattern)`.

Liên hệ với bài: với `pattern = nums`, `text = target`, ta có ngay `L_k = LCP(nums, W_k)` cho mọi `k` trong O(n + m).

In [11]:
nums = [2, 2, 3]
target = [2, 2, 3, 2, 1, 3, 2, 2]
n, m = len(nums), len(target)

s = nums + [0] + target
z = z_function(s)
print("s =", s)
print("z =", z)
for k in range(m - n + 1):
    print(f"k={k}: W_k={target[k:k + n]}, L_k = z[{n + 1 + k}] = {z[n + 1 + k]}")

s = [2, 2, 3, 0, 2, 2, 3, 2, 1, 3, 2, 2]
z = [12, 1, 0, 0, 3, 1, 0, 1, 0, 0, 2, 1]
k=0: W_k=[2, 2, 3], L_k = z[4] = 3
k=1: W_k=[2, 3, 2], L_k = z[5] = 1
k=2: W_k=[3, 2, 1], L_k = z[6] = 0
k=3: W_k=[2, 1, 3], L_k = z[7] = 1
k=4: W_k=[1, 3, 2], L_k = z[8] = 0
k=5: W_k=[3, 2, 2], L_k = z[9] = 0


# 4. Cách 1 — Brute force: làm đúng như đề nói

Ý tưởng tự nhiên nhất: **đọc đề và làm theo từng chữ**.

1. Duyệt mọi cửa sổ `k = 0..m-n`, lấy `W = target[k:k+n]`.
2. Với mỗi `i = 0..n-1`, dựng `A_i = nums[:i] + sorted(nums[i:], reverse=True)`.
3. Nếu `A_i == W` thì tăng biến đếm.

Cách này **chậm** nhưng **đúng hiển nhiên** — nó là "trọng tài" để kiểm tra mọi cách nhanh hơn ở phía sau.

In [12]:
def count_valid_suffix_sorts_bruteforce(nums: List[int], target: List[int]) -> List[int]:
    n, m = len(nums), len(target)
    ans = []
    for k in range(m - n + 1):
        window = target[k:k + n]
        count = 0
        for i in range(n):
            if sorted_suffix_array(nums, i) == window:
                count += 1
        ans.append(count)
    return ans

for nums, target, expected in examples:
    got = count_valid_suffix_sorts_bruteforce(nums, target)
    print(nums, target, "->", got, "| mong đợi", expected)
    assert got == expected

[2, 2, 3] [2, 2, 3, 2, 1, 3, 2, 2] -> [1, 1, 0, 0, 0, 1] | mong đợi [1, 1, 0, 0, 0, 1]
[5, 6, 5] [5, 6, 5, 5, 6, 5] -> [2, 1, 0, 2] | mong đợi [2, 1, 0, 2]


## Độ phức tạp của Cách 1

- Có `(m - n + 1) · n` cặp `(k, i)`.
- Mỗi cặp dựng `A_i`: slicing + sắp xếp `O(n log n)`, rồi so sánh `O(n)`.

```text
Time  = O((m - n + 1) · n · n log n)
Space = O(n)   (chỉ giữ một A_i và một cửa sổ tại mỗi thời điểm)
```

Với `n = 50 000`, `m = 100 000`, riêng phần "dựng và so sánh" (chưa tính `log n`) đã là `50 001 · 50 000 · 50 000 ≈ 1.25 · 10^14` thao tác — hoàn toàn không khả thi.

Có một sự lãng phí rất rõ: **`A_i` không phụ thuộc vào `k`**, vậy mà ta dựng lại nó cho **mỗi** cửa sổ. Cách 2 sửa đúng chỗ đó.

In [13]:
def estimate_work(n, m):
    windows = m - n + 1
    cach1 = windows * n * n          # mỗi (k, i): dựng + so sánh n phần tử (bỏ qua log n)
    cach2 = n * n + windows * n      # dựng n mảng một lần + đọc mỗi cửa sổ một lần
    return windows, cach1, cach2

for n, m in [(3, 8), (1_000, 2_000), (50_000, 100_000), (100_000, 100_000)]:
    w, c1, c2 = estimate_work(n, m)
    print(f"n={n:>7}, m={m:>7}: số cửa sổ={w:>6} | Cách 1 ~ {c1:.2e} | Cách 2 ~ {c2:.2e} phần tử phải đọc/ghi")

n=      3, m=      8: số cửa sổ=     6 | Cách 1 ~ 5.40e+01 | Cách 2 ~ 2.70e+01 phần tử phải đọc/ghi
n=   1000, m=   2000: số cửa sổ=  1001 | Cách 1 ~ 1.00e+09 | Cách 2 ~ 2.00e+06 phần tử phải đọc/ghi
n=  50000, m= 100000: số cửa sổ= 50001 | Cách 1 ~ 1.25e+14 | Cách 2 ~ 5.00e+09 phần tử phải đọc/ghi
n= 100000, m= 100000: số cửa sổ=     1 | Cách 1 ~ 1.00e+10 | Cách 2 ~ 1.00e+10 phần tử phải đọc/ghi


# 5. Cách 2 — Dựng các `A_i` một lần, gom bằng `Counter`

Hai quan sát từ Cách 1:

1. `A_0, ..., A_{n-1}` **không phụ thuộc** vào cửa sổ → dựng **một lần** thôi.
2. Nhiều `A_i` có thể **giống nhau** (Ví dụ 2: `A_1 = A_2`). Thay vì so cửa sổ với từng `A_i`, ta **đếm trước**:
   `freq[mảng] = số chỉ số i cho ra mảng đó`.

Khi đó `ans[k] = freq[tuple(W_k)]` (bằng `0` nếu không có) — một lần tra `dict`.

Cú pháp mới: `Counter(tuple(...) for i in range(n))` là một **generator expression** — giống list comprehension nhưng không tạo list trung gian; `Counter` đọc lần lượt từng phần tử.

In [14]:
def count_valid_suffix_sorts_counter(nums: List[int], target: List[int]) -> List[int]:
    n, m = len(nums), len(target)
    freq = Counter(tuple(sorted_suffix_array(nums, i)) for i in range(n))
    return [freq.get(tuple(target[k:k + n]), 0) for k in range(m - n + 1)]

freq_demo = Counter(tuple(sorted_suffix_array([5, 6, 5], i)) for i in range(3))
print("freq của Ví dụ 2:", dict(freq_demo))

for nums, target, expected in examples:
    got = count_valid_suffix_sorts_counter(nums, target)
    print(nums, target, "->", got, "| mong đợi", expected)
    assert got == expected

freq của Ví dụ 2: {(6, 5, 5): 1, (5, 6, 5): 2}
[2, 2, 3] [2, 2, 3, 2, 1, 3, 2, 2] -> [1, 1, 0, 0, 0, 1] | mong đợi [1, 1, 0, 0, 0, 1]
[5, 6, 5] [5, 6, 5, 5, 6, 5] -> [2, 1, 0, 2] | mong đợi [2, 1, 0, 2]


## Độ phức tạp của Cách 2 và "nút thắt" còn lại

```text
Time  = O(n² log n)          dựng n mảng, mỗi mảng sắp xếp O(n log n)
      + O((m - n + 1) · n)   mỗi cửa sổ: cắt ra tuple n phần tử + tính hash của tuple
Space = O(n²)                lưu tới n tuple, mỗi tuple n phần tử
```

Với `n = 10^5`: `n² = 10^10` phần tử phải lưu — không đủ bộ nhớ. Với `n = 50 000, m = 100 000`: tổng cộng cỡ `5 · 10^9` lần đọc/ghi (bảng ở cell trên), trong đó riêng phần tra cửa sổ đã là `50 001 · 50 000 ≈ 2.5 · 10^9`.

**Nút thắt:** mỗi câu hỏi "cửa sổ này bằng `A_i` nào?" vẫn đang tốn **O(n)** vì ta nhìn **toàn bộ** cửa sổ.

Muốn nhanh hơn, ta phải trả lời câu hỏi đó **mà không đọc lại cả cửa sổ**. Để làm được, cần hiểu sâu hơn **`A_i` trông như thế nào** — đó là bước đột phá.

# 6. Bước đột phá

## Đột phá 1 — Hình dạng của `A_i` cho ta 3 điều kiện

Nhìn lại cấu tạo: `A_i = nums[:i] + sorted(nums[i:], giảm dần)`. Mảng `A_i` có **hai phần**:

```text
A_i = [ nums[0], ..., nums[i-1] |  hậu tố đã sắp xếp không tăng  ]
        \____ y hệt nums ____/     \__ không tăng, cùng phần tử với nums[i:] __/
```

Vậy một cửa sổ `W` (độ dài `n`) bằng `A_i` khi và chỉ khi **cả 3 điều kiện** sau đúng:

- **(a) Tiền tố khớp:** `W[:i] == nums[:i]`.
- **(b) Đuôi không tăng:** `W[i:]` là dãy không tăng.
- **(c) Cùng multiset:** `W` là một hoán vị của `nums` (`Counter(W) == Counter(nums)`).

**Chiều thuận** (`W == A_i` ⇒ a, b, c): hiển nhiên từ cấu tạo của `A_i`.

**Chiều ngược** (a, b, c ⇒ `W == A_i`):

1. Từ (a) và (c): bỏ phần đầu giống nhau ra khỏi hai multiset bằng nhau, phần còn lại cũng bằng nhau → `W[i:]` và `nums[i:]` cùng multiset.
2. Một multiset chỉ có **đúng một** cách xếp thành dãy không tăng (chính là bản đã sắp xếp giảm dần).
3. Theo (b), `W[i:]` là dãy không tăng → `W[i:] == sorted(nums[i:], giảm dần)` → `W == A_i`.

Thí nghiệm dưới đây kiểm tra cả 3 điều kiện cho **mọi** cặp `(k, i)` của Ví dụ 2 và so với phép so sánh thật `A_i == W`.

In [15]:
def is_nonincreasing(a):
    return all(a[j] >= a[j + 1] for j in range(len(a) - 1))

nums = [5, 6, 5]
target = [5, 6, 5, 5, 6, 5]
n, m = len(nums), len(target)

print(" k  W          i | (a) tiền tố | (b) đuôi không tăng | (c) hoán vị | a&b&c | A_i == W")
for k in range(m - n + 1):
    W = target[k:k + n]
    for i in range(n):
        a = W[:i] == nums[:i]
        b = is_nonincreasing(W[i:])
        c = Counter(W) == Counter(nums)
        real = sorted_suffix_array(nums, i) == W
        assert (a and b and c) == real
        print(f" {k}  {W}  {i} | {a!s:^11} | {b!s:^19} | {c!s:^11} | {a and b and c!s:^5} | {real}")
print("3 điều kiện luôn trùng với phép so sánh thật.")

 k  W          i | (a) tiền tố | (b) đuôi không tăng | (c) hoán vị | a&b&c | A_i == W
 0  [5, 6, 5]  0 |    True     |        False        |    True     | False | False
 0  [5, 6, 5]  1 |    True     |        True         |    True     | True  | True
 0  [5, 6, 5]  2 |    True     |        True         |    True     | True  | True
 1  [6, 5, 5]  0 |    True     |        True         |    True     | True  | True
 1  [6, 5, 5]  1 |    False    |        True         |    True     | False | False
 1  [6, 5, 5]  2 |    False    |        True         |    True     | False | False
 2  [5, 5, 6]  0 |    True     |        False        |    True     | False | False
 2  [5, 5, 6]  1 |    True     |        False        |    True     | False | False
 2  [5, 5, 6]  2 |    False    |        True         |    True     | False | False
 3  [5, 6, 5]  0 |    True     |        False        |    True     | False | False
 3  [5, 6, 5]  1 |    True     |        True         |    True     | True  | True
 3  [

## Đột phá 2 — Tập các `i` thỏa (a) và tập các `i` thỏa (b) đều là **đoạn liên tục**

Cố định một cửa sổ `W_k`.

**Điều kiện (a)** `W[:i] == nums[:i]`: nếu đúng với `i` thì cũng đúng với mọi `i' < i` (tiền tố ngắn hơn vẫn khớp).
Gọi **`L_k` = LCP(nums, W_k)** (số phần tử đầu trùng nhau, `0 <= L_k <= n`). Khi đó:

> (a) đúng ⇔ `i <= L_k`. Kết hợp với `i <= n - 1`: **`i ∈ [0, min(L_k, n - 1)]`**.

**Điều kiện (b)** `W[i:]` không tăng: nếu đúng với `i` thì cũng đúng với mọi `i' > i` (hậu tố ngắn hơn của dãy không tăng vẫn không tăng).
Gọi **`s_k`** = chỉ số bắt đầu của **hậu tố không tăng dài nhất** của `W_k`. Khi đó:

> (b) đúng ⇔ **`i ∈ [s_k, n - 1]`**.

Hình dung trên trục `i`:

```text
i:        0   1   2   ...                 n-1
(a):      T   T   T   T   F   F   F   F        ← đúng ở đầu, tới min(L_k, n-1)
(b):      F   F   F   T   T   T   T   T        ← đúng ở cuối, từ s_k
(a)&(b):              T                         ← phần giao: [s_k, min(L_k, n-1)]
```

Thí nghiệm: in cờ (a), (b) theo `i` cho hai cửa sổ của `nums = [1, 2, 5, 4, 3, 3]`.

In [16]:
def show_flags(nums, W):
    n = len(nums)
    fa = ["T" if W[:i] == nums[:i] else "F" for i in range(n)]
    fb = ["T" if is_nonincreasing(W[i:]) else "F" for i in range(n)]
    both = [i for i in range(n) if fa[i] == "T" and fb[i] == "T"]
    print(f"W = {W}, hoán vị của nums? {Counter(W) == Counter(nums)}")
    print("   i  :", " ".join(str(i) for i in range(n)))
    print("  (a) :", " ".join(fa))
    print("  (b) :", " ".join(fb))
    print("  giao:", both, "-> số i hợp lệ =", len(both))
    print()

nums = [1, 2, 5, 4, 3, 3]
show_flags(nums, [1, 2, 5, 4, 3, 3])   # W == nums: L = 6, s = 2
show_flags(nums, [1, 2, 4, 5, 3, 3])   # hoán vị nhưng hai đoạn không giao nhau

W = [1, 2, 5, 4, 3, 3], hoán vị của nums? True
   i  : 0 1 2 3 4 5
  (a) : T T T T T T
  (b) : F F T T T T
  giao: [2, 3, 4, 5] -> số i hợp lệ = 4

W = [1, 2, 4, 5, 3, 3], hoán vị của nums? True
   i  : 0 1 2 3 4 5
  (a) : T T T F F F
  (b) : F F F T T T
  giao: [] -> số i hợp lệ = 0



## Đột phá 3 — Điều kiện (c) **không phụ thuộc vào `i`**

Điều kiện (c) chỉ hỏi "`W_k` có phải hoán vị của `nums`?" — chẳng liên quan gì đến `i`. Vậy với mỗi cửa sổ:

- Nếu (c) sai → **không** `i` nào hợp lệ → `ans[k] = 0`.
- Nếu (c) đúng → các `i` hợp lệ chính là đoạn giao `[s_k, min(L_k, n - 1)]`.

## Công thức chốt

```text
            ⎧ max(0, min(L_k, n - 1) - s_k + 1)   nếu W_k là hoán vị của nums
ans[k] =    ⎨
            ⎩ 0                                     nếu không
```

Kiểm tra nhanh với Ví dụ 2, cửa sổ `W_2 = [5, 5, 6]`: là hoán vị; `L_2 = 1` (chỉ số 5 đầu khớp); hậu tố không tăng dài nhất là `[6]` nên `s_2 = 2`.
Đoạn `[2, min(1, 2)] = [2, 1]` **rỗng** → `ans[2] = 0`. Đúng như đề!

Vì sao phải có `min(L_k, n - 1)`? Khi `W_k == nums` thì `L_k = n`, nhưng chỉ số `i` chỉ chạy tới `n - 1`. Quên chỗ này sẽ đếm dư 1 (Lỗi 2 ở mục 14).

Bây giờ ta viết một hàm dùng công thức, tính `L_k`, `s_k`, (c) **bằng cách quét thẳng** (chưa tối ưu) để kiểm chứng công thức trước đã.

In [17]:
def count_valid_conditions_naive(nums: List[int], target: List[int]) -> List[int]:
    n, m = len(nums), len(target)
    need = Counter(nums)
    ans = []
    for k in range(m - n + 1):
        W = target[k:k + n]
        if Counter(W) != need:                 # (c) sai -> 0
            ans.append(0)
            continue
        L = 0                                  # (a) L_k = LCP(nums, W)
        while L < n and W[L] == nums[L]:
            L += 1
        s = n - 1                              # (b) s_k: lùi khi còn W[s-1] >= W[s]
        while s > 0 and W[s - 1] >= W[s]:
            s -= 1
        ans.append(max(0, min(L, n - 1) - s + 1))
    return ans

for nums, target, expected in examples:
    got = count_valid_conditions_naive(nums, target)
    print(nums, target, "->", got, "| mong đợi", expected)
    assert got == expected

rng = random.Random(1)
for _ in range(300):
    n = rng.randint(1, 5)
    nums = [rng.randint(1, 3) for _ in range(n)]
    target = [rng.randint(1, 3) for _ in range(rng.randint(n, n + 6))]
    assert count_valid_conditions_naive(nums, target) == count_valid_suffix_sorts_bruteforce(nums, target)
print("Công thức khớp brute force trên 300 ca ngẫu nhiên.")

[2, 2, 3] [2, 2, 3, 2, 1, 3, 2, 2] -> [1, 1, 0, 0, 0, 1] | mong đợi [1, 1, 0, 0, 0, 1]
[5, 6, 5] [5, 6, 5, 5, 6, 5] -> [2, 1, 0, 2] | mong đợi [2, 1, 0, 2]
Công thức khớp brute force trên 300 ca ngẫu nhiên.


## Đột phá 4 — Kế hoạch tăng tốc từng mảnh

Công thức đã đúng, nhưng hàm trên vẫn quét `O(n)` cho mỗi cửa sổ → vẫn `O((m - n + 1) · n)`.
Điểm hay là bây giờ ta chỉ cần **3 con số cho mỗi cửa sổ**, và mỗi con số có một kỹ thuật kinh điển:

| Đại lượng | Ý nghĩa | Cách nhanh | Chi phí |
|---|---|---|---|
| `eq_k` | `W_k` là hoán vị của `nums`? | Sliding window + `diff` + `bad` (mục 3.3) | O(1) mỗi lần trượt |
| `s_k` | đầu hậu tố không tăng dài nhất của `W_k` | Mảng `last_up` của `target` (mục 3.4) | O(1) mỗi cửa sổ |
| `L_k` | LCP(nums, `W_k`) | Rolling hash + binary search (Cách 3) hoặc Z-function (Cách 4) | O(log n) hoặc O(1) |

Không còn chỗ nào phải "nhìn cả cửa sổ" nữa.

# 7. Cách 3 — Ba điều kiện + rolling hash + binary search

Ta lắp từng mảnh thành hàm riêng, test riêng, rồi mới ghép.

## 7.1 Mảnh `eq_k`: sliding window với mảng đếm

Giống hệt mục 3.3, nhưng vì giá trị `<= 10^5` ta dùng **list** `diff` kích thước `max_value + 1` thay cho `defaultdict` (nhanh hơn).

- Khởi tạo: cộng `+1` cho mỗi phần tử của `nums`, trừ `-1` cho mỗi phần tử của cửa sổ đầu `target[0:n]`.
- `bad` = số ô `diff` khác `0`.
- Trượt sang `k`: `target[k-1]` rời cửa sổ (`+1`), `target[k+n-1]` vào cửa sổ (`-1`).

Một trường hợp dễ lo: phần tử rời và phần tử vào **bằng nhau**. Khi đó `+1` rồi `-1` trên cùng ô — `bad` tăng rồi giảm lại, kết quả vẫn đúng.

In [18]:
def permutation_windows(nums: List[int], target: List[int]) -> List[bool]:
    """eq[k] = True nếu target[k:k+n] là một hoán vị của nums."""
    n, m = len(nums), len(target)
    diff = [0] * (max(max(nums), max(target)) + 1)
    for x in nums:
        diff[x] += 1
    for j in range(n):
        diff[target[j]] -= 1
    bad = sum(1 for d in diff if d != 0)
    eq = [bad == 0]
    for k in range(1, m - n + 1):
        for v, d in ((target[k - 1], +1), (target[k + n - 1], -1)):
            before = diff[v]
            diff[v] = before + d
            if before == 0:              # cân -> lệch
                bad += 1
            elif diff[v] == 0:           # lệch -> cân
                bad -= 1
        eq.append(bad == 0)
    return eq

print(permutation_windows([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]))
print(permutation_windows([5, 6, 5], [5, 6, 5, 5, 6, 5]))

[True, True, False, False, False, True]
[True, True, True, True]


## 7.2 Mảnh `s_k`: mảng `last_up` của `target`

Các cặp kề nhau **nằm trọn trong** cửa sổ `W_k = target[k..k+n-1]` là `(t, t+1)` với `t ∈ [k, k+n-2]`.

- Lấy `j = last_up[k + n - 2]` = vị trí đi lên cuối cùng `<= k + n - 2`.
- Nếu `j >= k` (vị trí đó nằm trong cửa sổ): hậu tố không tăng bắt đầu ngay sau nó, tính theo chỉ số **trong cửa sổ** là `s_k = j - k + 1`.
- Nếu `j < k` (không có vị trí đi lên nào trong cửa sổ): cả cửa sổ không tăng, `s_k = 0`.
- Nếu `n = 1`: cửa sổ chỉ 1 phần tử, `s_k = 0`.

In [19]:
def nonincreasing_starts(target: List[int], n: int) -> List[int]:
    """s[k] = chỉ số (trong cửa sổ) bắt đầu hậu tố không tăng dài nhất của target[k:k+n]."""
    m = len(target)
    last_up = [-1] * m
    prev = -1
    for t in range(m - 1):
        if target[t] < target[t + 1]:
            prev = t
        last_up[t] = prev
    starts = []
    for k in range(m - n + 1):
        j = last_up[k + n - 2] if n >= 2 else -1
        starts.append(j - k + 1 if j >= k else 0)
    return starts

target = [2, 2, 3, 2, 1, 3, 2, 2]
print("s =", nonincreasing_starts(target, 3))
# đối chiếu với cách quét thẳng
for k in range(len(target) - 3 + 1):
    W = target[k:k + 3]
    s = 2
    while s > 0 and W[s - 1] >= W[s]:
        s -= 1
    print(f"k={k}: W={W}, s quét thẳng = {s}, hậu tố = {W[s:]}")

s = [2, 1, 0, 2, 1, 0]
k=0: W=[2, 2, 3], s quét thẳng = 2, hậu tố = [3]
k=1: W=[2, 3, 2], s quét thẳng = 1, hậu tố = [3, 2]
k=2: W=[3, 2, 1], s quét thẳng = 0, hậu tố = [3, 2, 1]
k=3: W=[2, 1, 3], s quét thẳng = 2, hậu tố = [3]
k=4: W=[1, 3, 2], s quét thẳng = 1, hậu tố = [3, 2]
k=5: W=[3, 2, 2], s quét thẳng = 0, hậu tố = [3, 2, 2]


## 7.3 Mảnh `L_k`: binary search trên hash

Với mỗi `k`, tìm `l` lớn nhất trong `[0, n]` sao cho `hash(nums[0:l]) == hash(target[k:k+l])`.

- `hash(nums[0:l])` chính là `hn[l]` (prefix hash của `nums`).
- `hash(target[k:k+l])` là `get_hash(ht, pw, k, k + l)`.
- Điều kiện "khớp `l` phần tử đầu" là đơn điệu (mục 3.6) → binary search, mỗi bước O(1).

`k + l <= k + n <= m` nên không bao giờ vượt biên `target`.

In [20]:
def lcp_by_hash(nums: List[int], target: List[int]) -> List[int]:
    n, m = len(nums), len(target)
    pw = build_powers(n)
    hn = prefix_hash(nums)
    ht = prefix_hash(target)
    L = []
    for k in range(m - n + 1):
        lo, hi = 0, n
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if hn[mid] == get_hash(ht, pw, k, k + mid):
                lo = mid
            else:
                hi = mid - 1
        L.append(lo)
    return L

print("L (Ví dụ 1) =", lcp_by_hash([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]))
print("L (Ví dụ 2) =", lcp_by_hash([5, 6, 5], [5, 6, 5, 5, 6, 5]))

L (Ví dụ 1) = [3, 1, 0, 1, 0, 0]
L (Ví dụ 2) = [3, 0, 1, 3]


## 7.4 Ghép ba mảnh

Hàm `combine_answer` áp công thức của Đột phá 3 cho từng cửa sổ. Cả Cách 3 và Cách 4 đều dùng chung hàm này; chúng chỉ khác nhau ở cách tính `L`.

In [21]:
def combine_answer(n: int, eq: List[bool], L: List[int], s: List[int]) -> List[int]:
    ans = []
    for k in range(len(eq)):
        if not eq[k]:
            ans.append(0)
        else:
            hi = min(L[k], n - 1)
            ans.append(max(0, hi - s[k] + 1))
    return ans

def count_valid_suffix_sorts_hash(nums: List[int], target: List[int]) -> List[int]:
    n = len(nums)
    eq = permutation_windows(nums, target)      # (c)
    s = nonincreasing_starts(target, n)         # (b)
    L = lcp_by_hash(nums, target)               # (a)
    return combine_answer(n, eq, L, s)

for nums, target, expected in examples:
    got = count_valid_suffix_sorts_hash(nums, target)
    print(nums, target, "->", got, "| mong đợi", expected)
    assert got == expected

[2, 2, 3] [2, 2, 3, 2, 1, 3, 2, 2] -> [1, 1, 0, 0, 0, 1] | mong đợi [1, 1, 0, 0, 0, 1]
[5, 6, 5] [5, 6, 5, 5, 6, 5] -> [2, 1, 0, 2] | mong đợi [2, 1, 0, 2]


## Độ phức tạp của Cách 3

Gọi `V` là giá trị lớn nhất (`<= 10^5`).

```text
Time  = O(V)              tạo mảng diff
      + O(n + m)          prefix hash, last_up, sliding window
      + O((m - n + 1) · log n)   binary search cho mỗi cửa sổ
      = O(n + m · log n + V)
Space = O(n + m + V)
```

Với `m = 10^5`, `log₂ n ≈ 17` → khoảng `1.7 · 10^6` bước binary search: Python chạy thoải mái.

**Cải tiến nhỏ:** chỉ cần tính `L_k` khi `eq_k` đúng (nếu không, đáp án là `0` bất kể `L_k`). Bản LeetCode ở mục 10 làm đúng như vậy.

**Lưu ý trung thực:** hash là phương pháp **xác suất**. Với `MOD = 2^61 - 1`, xác suất va chạm trong cả bài là cực nhỏ,
nhưng một `BASE` **cố định** về lý thuyết có thể bị dựng test "anti-hash". Muốn an toàn hơn: chọn `BASE` ngẫu nhiên lúc chạy,
dùng 2 hash, hoặc — tốt nhất — dùng Z-function **chính xác tuyệt đối** ở Cách 4.

# 8. Cách 4 — Ba điều kiện + Z-function (O(n + m), không xác suất)

Chỉ thay mảnh `L_k`:

1. Ghép `s = nums + [0] + target` (độ dài `n + 1 + m`).
2. Chạy `z_function(s)` một lần — O(n + m).
3. `L_k = z[n + 1 + k]`, vì vị trí `n + 1 + k` trong `s` chính là `target[k]`.

Số `0` không xuất hiện trong `nums` lẫn `target`, nên phép khớp dừng lại trước khi vượt quá `n` phần tử → `L_k <= n` tự động.

Ta kiểm tra `lcp_by_z` và `lcp_by_hash` cho cùng kết quả, rồi ghép thành lời giải hoàn chỉnh.

In [22]:
def lcp_by_z(nums: List[int], target: List[int]) -> List[int]:
    n, m = len(nums), len(target)
    z = z_function(nums + [0] + target)
    return [z[n + 1 + k] for k in range(m - n + 1)]

rng = random.Random(2)
for _ in range(300):
    n = rng.randint(1, 6)
    nums = [rng.randint(1, 3) for _ in range(n)]
    target = [rng.randint(1, 3) for _ in range(rng.randint(n, n + 8))]
    assert lcp_by_z(nums, target) == lcp_by_hash(nums, target)
print("lcp_by_z khớp lcp_by_hash trên 300 ca ngẫu nhiên")
print("L (Ví dụ 2) =", lcp_by_z([5, 6, 5], [5, 6, 5, 5, 6, 5]))

lcp_by_z khớp lcp_by_hash trên 300 ca ngẫu nhiên
L (Ví dụ 2) = [3, 0, 1, 3]


In [23]:
def count_valid_suffix_sorts_z(nums: List[int], target: List[int]) -> List[int]:
    n = len(nums)
    eq = permutation_windows(nums, target)      # (c) hoán vị?
    s = nonincreasing_starts(target, n)         # (b) đầu hậu tố không tăng
    L = lcp_by_z(nums, target)                  # (a) LCP với nums
    return combine_answer(n, eq, L, s)

for nums, target, expected in examples:
    got = count_valid_suffix_sorts_z(nums, target)
    print(nums, target, "->", got, "| mong đợi", expected)
    assert got == expected

[2, 2, 3] [2, 2, 3, 2, 1, 3, 2, 2] -> [1, 1, 0, 0, 0, 1] | mong đợi [1, 1, 0, 0, 0, 1]
[5, 6, 5] [5, 6, 5, 5, 6, 5] -> [2, 1, 0, 2] | mong đợi [2, 1, 0, 2]


## Độ phức tạp của Cách 4

```text
Time  = O(n + m + V)     Z-function O(n + m), last_up O(m), sliding window O(n + m), mảng diff O(V)
Space = O(n + m + V)
```

Đây là mức tối ưu về mặt tiệm cận: chỉ riêng việc **đọc** input đã tốn O(n + m).
(Nếu thay mảng `diff` bằng `dict`, phần `V` biến mất: O(n + m) thời gian trung bình.)

# 9. Mô phỏng từng bước

## 9.1 Bảng trạng thái của Cách 4

Dùng một ví dụ "đủ khó" để thấy **mọi nhánh**: `nums = [3, 1, 2]`, `target = [3, 2, 1, 3, 1, 2, 2, 1, 3]`.

Các mảng: `A_0 = [3, 2, 1]`, `A_1 = [3, 2, 1]` (trùng `A_0`!), `A_2 = [3, 1, 2]`.

| k | `W_k` | hoán vị? (`bad == 0`) | `L_k` | `s_k` | đoạn `[s_k, min(L_k, n-1)]` | `ans[k]` |
|---|---|---|---|---|---|---|
| 0 | `[3, 2, 1]` | có | 1 | 0 | `[0, 1]` | 2 |
| 1 | `[2, 1, 3]` | có | 0 | 2 | `[2, 0]` rỗng | 0 |
| 2 | `[1, 3, 1]` | không | — | — | loại ngay | 0 |
| 3 | `[3, 1, 2]` | có | 3 | 2 | `[2, 2]` | 1 |
| 4 | `[1, 2, 2]` | không | — | — | loại ngay | 0 |
| 5 | `[2, 2, 1]` | không | — | — | loại ngay | 0 |
| 6 | `[2, 1, 3]` | có | 0 | 2 | `[2, 0]` rỗng | 0 |

Đọc bảng:

- `k = 0`: hai chỉ số `i = 0, 1` cùng tạo ra `[3, 2, 1]` → đáp án 2 (bài học "đếm chỉ số").
- `k = 1, 6`: là hoán vị, nhưng tiền tố khớp quá ngắn (`L = 0`) còn đuôi không tăng quá ngắn (`s = 2`) → hai đoạn không gặp nhau.
- `k = 3`: cửa sổ **bằng đúng `nums`** nên `L = 3 = n`, phải cắt về `n - 1 = 2`.

Hàm `trace_...` dưới đây in đúng các cột đó (kèm phần tử rời/vào cửa sổ) để bạn đối chiếu.

In [24]:
def trace_count_valid_suffix_sorts(nums, target):
    n, m = len(nums), len(target)
    eq = permutation_windows(nums, target)
    s = nonincreasing_starts(target, n)
    L = lcp_by_z(nums, target)
    print(f"nums = {nums}, n = {n}")
    for i in range(n):
        print(f"  A_{i} = {sorted_suffix_array(nums, i)}")
    print("-" * 72)
    ans = []
    for k in range(m - n + 1):
        move = "cửa sổ đầu" if k == 0 else f"rời {target[k - 1]}, vào {target[k + n - 1]}"
        head = f"k={k} ({move}): W={target[k:k + n]}"
        if not eq[k]:
            ans.append(0)
            print(f"{head} | bad>0 -> không phải hoán vị -> ans[{k}]=0")
            continue
        hi = min(L[k], n - 1)
        cnt = max(0, hi - s[k] + 1)
        ans.append(cnt)
        seg = f"[{s[k]}, {hi}]" + ("" if cnt else " rỗng")
        print(f"{head} | bad=0, L={L[k]}, s={s[k]}, đoạn {seg} -> ans[{k}]={cnt}")
    print("ans =", ans)
    return ans

trace_result = trace_count_valid_suffix_sorts([3, 1, 2], [3, 2, 1, 3, 1, 2, 2, 1, 3])
assert trace_result == count_valid_suffix_sorts_bruteforce([3, 1, 2], [3, 2, 1, 3, 1, 2, 2, 1, 3])

nums = [3, 1, 2], n = 3
  A_0 = [3, 2, 1]
  A_1 = [3, 2, 1]
  A_2 = [3, 1, 2]
------------------------------------------------------------------------
k=0 (cửa sổ đầu): W=[3, 2, 1] | bad=0, L=1, s=0, đoạn [0, 1] -> ans[0]=2
k=1 (rời 3, vào 3): W=[2, 1, 3] | bad=0, L=0, s=2, đoạn [2, 0] rỗng -> ans[1]=0
k=2 (rời 2, vào 1): W=[1, 3, 1] | bad>0 -> không phải hoán vị -> ans[2]=0
k=3 (rời 1, vào 2): W=[3, 1, 2] | bad=0, L=3, s=2, đoạn [2, 2] -> ans[3]=1
k=4 (rời 3, vào 2): W=[1, 2, 2] | bad>0 -> không phải hoán vị -> ans[4]=0
k=5 (rời 1, vào 1): W=[2, 2, 1] | bad>0 -> không phải hoán vị -> ans[5]=0
k=6 (rời 2, vào 3): W=[2, 1, 3] | bad=0, L=0, s=2, đoạn [2, 0] rỗng -> ans[6]=0
ans = [2, 0, 0, 1, 0, 0, 0]


## 9.2 Mô phỏng Z-function trên `s = nums + [0] + target` của Ví dụ 1

`s = [2, 2, 3, 0, 2, 2, 3, 2, 1, 3, 2, 2]` (chỉ số 0..11). Các cột: hộp `[l, r)` **trước** khi xử lý `i`, giá trị khởi đầu lấy từ hộp, số lần so sánh thêm thành công, `z[i]` cuối cùng.

| i | `s[i]` | hộp trước | khởi đầu từ hộp | so sánh thêm | `z[i]` | hộp sau |
|---|---|---|---|---|---|---|
| 1 | 2 | `[0, 0)` | 0 | 1 | 1 | `[1, 2)` |
| 2 | 3 | `[1, 2)` | 0 | 0 | 0 | `[1, 2)` |
| 3 | 0 | `[1, 2)` | 0 | 0 | 0 | `[3, 3)` (rỗng) |
| 4 | 2 | `[3, 3)` | 0 | 3 | 3 | `[4, 7)` |
| 5 | 2 | `[4, 7)` | `min(2, z[1]) = 1` | 0 | 1 | `[4, 7)` |
| 6 | 3 | `[4, 7)` | `min(1, z[2]) = 0` | 0 | 0 | `[4, 7)` |
| 7 | 2 | `[4, 7)` | 0 | 1 | 1 | `[7, 8)` |
| 8 | 1 | `[7, 8)` | 0 | 0 | 0 | `[7, 8)` |
| 9 | 3 | `[7, 8)` | 0 | 0 | 0 | `[9, 9)` (rỗng) |
| 10 | 2 | `[9, 9)` | 0 | 2 | 2 | `[10, 12)` |
| 11 | 2 | `[10, 12)` | `min(1, z[1]) = 1` | 0 | 1 | `[10, 12)` |

Ở `i = 5` và `i = 11`, giá trị được **lấy lại từ hộp** mà không cần so sánh — đó là lý do Z-function chạy O(N).

Ở `i = 3` và `i = 9`: `z[i] = 0` nhưng `i + z[i] = i > r`, nên code vẫn "cập nhật" thành hộp **rỗng** `[i, i)`.
Điều này vô hại: hộp rỗng không chứa chỉ số nào, nên ở bước sau điều kiện `i < r` chỉ đúng khi hộp thật sự có nội dung.
Các giá trị `z[4..9]` = `3, 1, 0, 1, 0, 0` chính là `L_0..L_5` của Ví dụ 1 (cell mục 3.7 đã in ra).

In [25]:
def trace_z_function(s):
    N = len(s)
    z = [0] * N
    z[0] = N
    l, r = 0, 0
    print(f"s = {s}")
    for i in range(1, N):
        box_before = f"[{l}, {r})"
        if i < r:
            z[i] = min(r - i, z[i - l])
        start = z[i]
        while i + z[i] < N and s[z[i]] == s[i + z[i]]:
            z[i] += 1
        if i + z[i] > r:
            l, r = i, i + z[i]
        print(f"i={i:>2}, s[i]={s[i]}, hộp trước={box_before:>8}, khởi đầu={start}, "
              f"so sánh thêm={z[i] - start}, z[i]={z[i]}, hộp sau=[{l}, {r})")
    return z

z_trace = trace_z_function([2, 2, 3] + [0] + [2, 2, 3, 2, 1, 3, 2, 2])
assert z_trace == z_function([2, 2, 3, 0, 2, 2, 3, 2, 1, 3, 2, 2])

s = [2, 2, 3, 0, 2, 2, 3, 2, 1, 3, 2, 2]
i= 1, s[i]=2, hộp trước=  [0, 0), khởi đầu=0, so sánh thêm=1, z[i]=1, hộp sau=[1, 2)
i= 2, s[i]=3, hộp trước=  [1, 2), khởi đầu=0, so sánh thêm=0, z[i]=0, hộp sau=[1, 2)
i= 3, s[i]=0, hộp trước=  [1, 2), khởi đầu=0, so sánh thêm=0, z[i]=0, hộp sau=[3, 3)
i= 4, s[i]=2, hộp trước=  [3, 3), khởi đầu=0, so sánh thêm=3, z[i]=3, hộp sau=[4, 7)
i= 5, s[i]=2, hộp trước=  [4, 7), khởi đầu=1, so sánh thêm=0, z[i]=1, hộp sau=[4, 7)
i= 6, s[i]=3, hộp trước=  [4, 7), khởi đầu=0, so sánh thêm=0, z[i]=0, hộp sau=[4, 7)
i= 7, s[i]=2, hộp trước=  [4, 7), khởi đầu=0, so sánh thêm=1, z[i]=1, hộp sau=[7, 8)
i= 8, s[i]=1, hộp trước=  [7, 8), khởi đầu=0, so sánh thêm=0, z[i]=0, hộp sau=[7, 8)
i= 9, s[i]=3, hộp trước=  [7, 8), khởi đầu=0, so sánh thêm=0, z[i]=0, hộp sau=[9, 9)
i=10, s[i]=2, hộp trước=  [9, 9), khởi đầu=0, so sánh thêm=2, z[i]=2, hộp sau=[10, 12)
i=11, s[i]=2, hộp trước=[10, 12), khởi đầu=1, so sánh thêm=0, z[i]=1, hộp sau=[10, 12)


# 10. Phiên bản nộp LeetCode

Khi nộp bài:

- Viết trong `class Solution`, method `countValidSuffixSorts(self, nums, target)` (xem ghi chú tên method ở đầu notebook).
- Cần `from typing import List` nếu chạy ngoài LeetCode.
- **Không** `print` trong code nộp.
- Gộp các mảnh lại trong một hàm để Python chạy nhanh hơn (ít lời gọi hàm), và **chỉ tính `s_k`, đọc `L_k` khi cửa sổ là hoán vị**.

Code dưới dùng Cách 4 (Z-function) vì nó chính xác tuyệt đối và O(n + m).

In [26]:
from typing import List

class Solution:
    def countValidSuffixSorts(self, nums: List[int], target: List[int]) -> List[int]:
        n, m = len(nums), len(target)

        # (a) Z-function trên nums + [0] + target  ->  L_k = z[n + 1 + k]
        s = nums + [0] + target
        N = len(s)
        z = [0] * N
        l = r = 0
        for i in range(1, N):
            if i < r:
                z[i] = min(r - i, z[i - l])
            while i + z[i] < N and s[z[i]] == s[i + z[i]]:
                z[i] += 1
            if i + z[i] > r:
                l, r = i, i + z[i]

        # (b) last_up[t] = vị trí đi lên (target[t] < target[t+1]) gần nhất <= t
        last_up = [-1] * m
        prev = -1
        for t in range(m - 1):
            if target[t] < target[t + 1]:
                prev = t
            last_up[t] = prev

        # (c) sliding window: diff[v] = #v trong nums - #v trong cửa sổ
        diff = [0] * (max(max(nums), max(target)) + 1)
        for x in nums:
            diff[x] += 1
        for j in range(n):
            diff[target[j]] -= 1
        bad = sum(1 for d in diff if d)

        ans = [0] * (m - n + 1)
        for k in range(m - n + 1):
            if k > 0:
                v = target[k - 1]                 # rời cửa sổ
                if diff[v] == 0:
                    bad += 1
                diff[v] += 1
                if diff[v] == 0:
                    bad -= 1
                v = target[k + n - 1]             # vào cửa sổ
                if diff[v] == 0:
                    bad += 1
                diff[v] -= 1
                if diff[v] == 0:
                    bad -= 1
            if bad:
                continue
            hi = min(z[n + 1 + k], n - 1)
            j = last_up[k + n - 2] if n >= 2 else -1
            lo = j - k + 1 if j >= k else 0
            if hi >= lo:
                ans[k] = hi - lo + 1
        return ans

In [27]:
sol = Solution()

# Ví dụ của đề
assert sol.countValidSuffixSorts([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]) == [1, 1, 0, 0, 0, 1]
assert sol.countValidSuffixSorts([5, 6, 5], [5, 6, 5, 5, 6, 5]) == [2, 1, 0, 2]

# Edge cases: (nums, target, expected)
edge_cases = [
    ([5], [5, 1, 5], [1, 0, 1]),               # n = 1: A_0 = nums
    ([1, 2, 3], [3, 2, 1], [1]),               # n = m, khớp A_0
    ([1, 2, 3], [1, 2, 3], [1]),               # n = m, khớp A_2 = nums
    ([7, 7, 7], [7, 7, 7, 7], [3, 3]),         # toàn bằng nhau: mọi A_i giống nhau
    ([3, 2, 1], [3, 2, 1], [3]),               # nums đã không tăng: mọi A_i = nums
    ([1, 2], [3, 3, 3], [0, 0]),               # không cửa sổ nào là hoán vị
    ([2, 1], [2, 1, 3], [2, 0]),               # cặp đi lên ngay ngoài mép cửa sổ
    ([1, 2], [1, 3], [0]),                     # tiền tố + đuôi đều ổn nhưng không phải hoán vị
]
for nums, target, expected in edge_cases:
    got = sol.countValidSuffixSorts(nums, target)
    assert got == expected == count_valid_suffix_sorts_bruteforce(nums, target), (nums, target, got)
    print(f"PASS  nums={nums}, target={target} -> {got}")

PASS  nums=[5], target=[5, 1, 5] -> [1, 0, 1]
PASS  nums=[1, 2, 3], target=[3, 2, 1] -> [1]
PASS  nums=[1, 2, 3], target=[1, 2, 3] -> [1]
PASS  nums=[7, 7, 7], target=[7, 7, 7, 7] -> [3, 3]
PASS  nums=[3, 2, 1], target=[3, 2, 1] -> [3]
PASS  nums=[1, 2], target=[3, 3, 3] -> [0, 0]
PASS  nums=[2, 1], target=[2, 1, 3] -> [2, 0]
PASS  nums=[1, 2], target=[1, 3] -> [0]


In [28]:
import time

rng_big = random.Random(2026)
big_tests = [
    ("n = m = 10^5, toàn số 7", [7] * 100_000, [7] * 100_000),
    ("n = 5·10^4, m = 10^5, giá trị 1..3", [rng_big.randint(1, 3) for _ in range(50_000)], [rng_big.randint(1, 3) for _ in range(100_000)]),
    ("n = 5·10^4, nums = [1,2]*, target = [1,2]*", [1, 2] * 25_000, [1, 2] * 50_000),
    ("n = 1, m = 10^5", [100_000], [100_000] * 100_000),
]
for name, nums, target in big_tests:
    t0 = time.perf_counter()
    res = sol.countValidSuffixSorts(nums, target)
    dt = time.perf_counter() - t0
    print(f"{name:<45} len(ans)={len(res):>6}, tổng ans={sum(res):>7}, thời gian={dt:.3f}s")

n = m = 10^5, toàn số 7                       len(ans)=     1, tổng ans= 100000, thời gian=0.054s


n = 5·10^4, m = 10^5, giá trị 1..3            len(ans)= 50001, tổng ans=      0, thời gian=0.042s


n = 5·10^4, nums = [1,2]*, target = [1,2]*    len(ans)= 50001, tổng ans=  25001, thời gian=0.058s


n = 1, m = 10^5                               len(ans)=100000, tổng ans= 100000, thời gian=0.051s


# 11. Kiểm tra chéo tất cả các cách

Mọi cách nhanh đều được so với brute force (trọng tài) trên nhiều input nhỏ ngẫu nhiên, **seed cố định** để kết quả lặp lại được.

Mẹo tạo test tốt: nếu chỉ sinh `target` ngẫu nhiên, gần như không cửa sổ nào bằng một `A_i` → đáp án toàn `0`, test "dễ".
Vì vậy một nửa số ca ta **cố ý nhúng** một `A_i` vào `target`, và dùng ít giá trị khác nhau (1..4) để có nhiều phần tử trùng.

In [29]:
rng = random.Random(4109)
approaches = {
    "Cách 2 (Counter)": count_valid_suffix_sorts_counter,
    "Điều kiện, quét thẳng": count_valid_conditions_naive,
    "Cách 3 (hash)": count_valid_suffix_sorts_hash,
    "Cách 4 (Z)": count_valid_suffix_sorts_z,
    "Solution": Solution().countValidSuffixSorts,
}
cases = nonzero = 0
for _ in range(400):
    n = rng.randint(1, 6)
    m = rng.randint(n, n + 7)
    V = rng.randint(1, 4)
    nums = [rng.randint(1, V) for _ in range(n)]
    target = [rng.randint(1, V) for _ in range(m)]
    if rng.random() < 0.5:                       # nhúng một A_i vào target
        p = rng.randint(0, m - n)
        target[p:p + n] = sorted_suffix_array(nums, rng.randrange(n))
    expected = count_valid_suffix_sorts_bruteforce(nums, target)
    for name, f in approaches.items():
        assert f(nums, target) == expected, (name, nums, target)
    cases += 1
    nonzero += any(expected)
print(f"Đã kiểm tra {cases} ca, {nonzero} ca có đáp án khác 0 — tất cả {len(approaches)} cách khớp brute force.")

Đã kiểm tra 400 ca, 302 ca có đáp án khác 0 — tất cả 5 cách khớp brute force.


# 12. Vì sao thuật toán đúng?

## Bổ đề 1 — Ba điều kiện (đã chứng minh ở Đột phá 1)

`W == A_i` ⇔ (a) `W[:i] == nums[:i]` **và** (b) `W[i:]` không tăng **và** (c) `W` là hoán vị của `nums`.

Mấu chốt của chiều ngược: một multiset chỉ có **một** cách xếp không tăng, nên "không tăng + đúng phần tử" là đủ để khẳng định "bằng bản đã sắp xếp".

## Bổ đề 2 — Hai đoạn liên tục

- Tập `{i : (a) đúng}` = `{0, 1, ..., min(L_k, n - 1)}` vì tiền tố khớp thì mọi tiền tố ngắn hơn cũng khớp.
- Tập `{i : (b) đúng}` = `{s_k, ..., n - 1}` vì hậu tố của dãy không tăng cũng không tăng, và `s_k` được định nghĩa là chỗ bắt đầu sớm nhất.
- (c) không phụ thuộc `i`.

⇒ Số `i` hợp lệ = độ dài đoạn giao `[s_k, min(L_k, n - 1)]` nếu (c) đúng, ngược lại `0`. Không bỏ sót `i` nào, không đếm nhầm `i` nào.

## Bất biến (invariant) của từng mảnh

- **Sliding window:** sau khi xử lý cửa sổ `k`, với mọi `v`: `diff[v] = #v(nums) - #v(W_k)`, và `bad` = số `v` có `diff[v] != 0`.
  Mỗi lần trượt chỉ 2 ô thay đổi và `bad` được chỉnh đúng theo 3 trường hợp (cân→lệch, lệch→cân, lệch→lệch). Vậy (c) ⇔ `bad == 0`.
- **`last_up`:** `last_up[t]` = vị trí đi lên lớn nhất `<= t`. Cặp kề cuối cùng nằm trong `W_k` là `(k+n-2, k+n-1)`, nên vị trí đi lên cuối cùng **trong** cửa sổ là `last_up[k+n-2]` nếu nó `>= k`.
- **Z-function:** hộp `[l, r)` luôn thỏa `s[l:r] == s[0:r-l]`; nhờ đó giá trị khởi đầu `min(r - i, z[i - l])` là một **cận dưới đúng** của `z[i]`, và vòng `while` kéo dài tới đúng chỗ khác nhau đầu tiên.
  Separator `0` không có trong input nên `z[n + 1 + k] <= n` và bằng đúng `LCP(nums, target[k:])`.
- **Hash (Cách 3):** hai đoạn bằng nhau ⇒ hash bằng nhau (luôn đúng). Chiều ngược chỉ đúng **với xác suất rất cao**; binary search hợp lệ vì điều kiện khớp tiền tố là đơn điệu.

# 13. Độ phức tạp và so sánh các cách

Ký hiệu: `n = len(nums)`, `m = len(target)`, số cửa sổ `w = m - n + 1`, `V` = giá trị lớn nhất (`<= 10^5`).

| Cách | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|
| 1. Brute force | O(w · n² · log n) | O(n) | Trọng tài; chỉ dùng cho input nhỏ |
| 2. `Counter` các tuple | O(n² · log n + w · n) | O(n²) | Bỏ được việc dựng lại `A_i`, nhưng vẫn đọc cả cửa sổ |
| (Kiểm chứng) Điều kiện quét thẳng | O(w · n) | O(n) | Không sắp xếp, nhưng vẫn quét cả cửa sổ |
| 3. Rolling hash + binary search | O(n + m · log n + V) | O(n + m + V) | Xác suất (rủi ro va chạm cực nhỏ) |
| 4. Z-function | **O(n + m + V)** | O(n + m + V) | Chính xác, nhanh nhất — bản nộp bài |

Với `n, m <= 10^5`, Cách 3 và Cách 4 đều chạy tốt trong Python thuần (cell thời gian ở mục 10 cho thấy cỡ phần nhỏ của một giây trên máy chạy notebook).
Nếu môi trường chấm chậm hơn, Cách 4 bản "gộp một hàm" vẫn là lựa chọn an toàn nhất vì chỉ có vài vòng lặp tuyến tính.

# 14. Những lỗi dễ mắc

## Lỗi 1 — Quên kiểm tra hoán vị (điều kiện c)

Sai: chỉ dùng `L_k` và `s_k`.

```python
ans[k] = max(0, min(L[k], n - 1) - s[k] + 1)      # thiếu "if eq[k]"
```

Phản ví dụ: `nums = [1, 2]`, `target = [1, 3]`. Tiền tố khớp `[1]` (`L = 1`), đuôi `[3]` không tăng (`s = 1`) → công thức đếm `i = 1`.
Nhưng `A_1 = [1, 2] != [1, 3]`. Có số `3` lạ mà (a), (b) không phát hiện được.

## Lỗi 2 — Quên chặn `L_k` ở `n - 1`

Khi cửa sổ **bằng đúng `nums`**, `L_k = n`, nhưng `i` chỉ đi tới `n - 1`. Dùng `L_k` thay cho `min(L_k, n - 1)` sẽ đếm dư 1.
Phản ví dụ: `nums = [2, 1]`, `target = [2, 1]` → đúng là `2` (`A_0 = A_1 = [2, 1]`), sai thành `3`.

## Lỗi 3 — Dùng `>` (giảm ngặt) thay cho `>=` (không tăng)

Đề nói **non-increasing**: được phép bằng nhau. Nếu viết `while s > 0 and W[s-1] > W[s]`, các phần tử trùng sẽ làm `s_k` lớn hơn thực tế.
Phản ví dụ: `nums = [2, 2]`, `target = [2, 2]` → đúng là `2`, sai thành `1`.

In [30]:
def wrong_no_permutation_check(nums, target):
    n = len(nums)
    s = nonincreasing_starts(target, n)
    L = lcp_by_z(nums, target)
    return [max(0, min(L[k], n - 1) - s[k] + 1) for k in range(len(target) - n + 1)]

def wrong_no_cap(nums, target):
    n = len(nums)
    eq, s, L = permutation_windows(nums, target), nonincreasing_starts(target, n), lcp_by_z(nums, target)
    return [max(0, L[k] - s[k] + 1) if eq[k] else 0 for k in range(len(eq))]

def wrong_strict(nums, target):
    n, m = len(nums), len(target)
    ans = []
    for k in range(m - n + 1):
        W = target[k:k + n]
        if Counter(W) != Counter(nums):
            ans.append(0)
            continue
        L = 0
        while L < n and W[L] == nums[L]:
            L += 1
        s = n - 1
        while s > 0 and W[s - 1] > W[s]:        # SAI: phải là >=
            s -= 1
        ans.append(max(0, min(L, n - 1) - s + 1))
    return ans

print("Lỗi 1:", wrong_no_permutation_check([1, 2], [1, 3]), "| đúng:", count_valid_suffix_sorts_z([1, 2], [1, 3]))
print("Lỗi 2:", wrong_no_cap([2, 1], [2, 1]), "| đúng:", count_valid_suffix_sorts_z([2, 1], [2, 1]))
print("Lỗi 3:", wrong_strict([2, 2], [2, 2]), "| đúng:", count_valid_suffix_sorts_z([2, 2], [2, 2]))

Lỗi 1: [1] | đúng: [0]
Lỗi 2: [3] | đúng: [2]
Lỗi 3: [1] | đúng: [2]


## Lỗi 4 — Đếm số **mảng khác nhau** thay vì số **chỉ số**

Nếu gom các `A_i` vào `set` (chỉ biết "có hay không"), Ví dụ 2 sẽ ra `[1, 1, 0, 1]` thay vì `[2, 1, 0, 2]`.
Phải dùng `Counter` (đếm số lần), hoặc như Cách 4 — đếm độ dài đoạn `i`.

## Lỗi 5 — Lệch một chỉ số khi tra `last_up`

Cặp kề cuối cùng **trong** cửa sổ là `(k+n-2, k+n-1)`, nên phải tra `last_up[k + n - 2]`.
Tra `last_up[k + n - 1]` là đã tính cả cặp `(k+n-1, k+n)` — phần tử `target[k+n]` **nằm ngoài** cửa sổ.
Phản ví dụ: `nums = [2, 1]`, `target = [2, 1, 3]`: cửa sổ `[2, 1]` bị "vạ lây" vì `1 < 3` ở ngoài mép → đáp án `2` bị biến thành `0`.

## Lỗi 6 — Hash yếu: `BASE` nhỏ hơn giá trị phần tử

Với `BASE = 10`: `hash([1, 11]) = 1·10 + 11 = 21 = 2·10 + 1 = hash([2, 1])` — **va chạm**, dù không hề có modulo!
Lý do: "chữ số" `11` lớn hơn cơ số nên tràn sang hàng bên cạnh. Phản ví dụ dưới đây làm LCP bị tính sai và đáp án sai theo.
Sửa: `BASE > max giá trị` (ở đây `1_000_003 > 10^5`) **và** `MOD` thật lớn; hoặc dùng Z-function.

In [31]:
# Lỗi 4: dùng set
def wrong_distinct_arrays(nums, target):
    n = len(nums)
    seen = {tuple(sorted_suffix_array(nums, i)) for i in range(n)}
    return [1 if tuple(target[k:k + n]) in seen else 0 for k in range(len(target) - n + 1)]
print("Lỗi 4:", wrong_distinct_arrays([5, 6, 5], [5, 6, 5, 5, 6, 5]), "| đúng:", count_valid_suffix_sorts_z([5, 6, 5], [5, 6, 5, 5, 6, 5]))

# Lỗi 5: tra last_up lệch một chỉ số
def wrong_last_up_index(nums, target):
    n, m = len(nums), len(target)
    eq, L = permutation_windows(nums, target), lcp_by_z(nums, target)
    last_up = [-1] * m
    prev = -1
    for t in range(m - 1):
        if target[t] < target[t + 1]:
            prev = t
        last_up[t] = prev
    ans = []
    for k in range(m - n + 1):
        j = last_up[k + n - 1]                 # SAI: phải là k + n - 2
        s = j - k + 1 if j >= k else 0
        ans.append(max(0, min(L[k], n - 1) - s + 1) if eq[k] else 0)
    return ans
print("Lỗi 5:", wrong_last_up_index([2, 1], [2, 1, 3]), "| đúng:", count_valid_suffix_sorts_z([2, 1], [2, 1, 3]))

# Lỗi 6: hash với BASE = 10
def weak_hash(a):
    h = 0
    for x in a:
        h = (h * 10 + x) % (10**9 + 7)
    return h
print("weak_hash([1, 11]) =", weak_hash([1, 11]), "; weak_hash([2, 1]) =", weak_hash([2, 1]))

def wrong_weak_hash_lcp(nums, target):
    n, m = len(nums), len(target)
    L = []
    for k in range(m - n + 1):
        lo, hi = 0, n
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if weak_hash(nums[:mid]) == weak_hash(target[k:k + mid]):
                lo = mid
            else:
                hi = mid - 1
        L.append(lo)
    return combine_answer(n, permutation_windows(nums, target), L, nonincreasing_starts(target, n))
print("Lỗi 6:", wrong_weak_hash_lcp([2, 1, 1, 11], [1, 11, 2, 1]), "| đúng:", count_valid_suffix_sorts_z([2, 1, 1, 11], [1, 11, 2, 1]))

Lỗi 4: [1, 1, 0, 1] | đúng: [2, 1, 0, 2]
Lỗi 5: [0, 0] | đúng: [2, 0]
weak_hash([1, 11]) = 21 ; weak_hash([2, 1]) = 21
Lỗi 6: [2] | đúng: [0]


## Edge cases nên tự kiểm tra

| Tình huống | Ví dụ | Vì sao dễ sai |
|---|---|---|
| `n = 1` | `nums = [5]`, `target = [5, 1, 5]` → `[1, 0, 1]` | Không có cặp kề nào; không được tra `last_up[k - 1]` |
| `n = m` | chỉ có 1 cửa sổ | Vòng trượt không chạy lần nào |
| Toàn bằng nhau | `[7, 7, 7]` vs `[7, 7, 7, 7]` → `[3, 3]` | Mọi `A_i` trùng nhau; phân biệt `>` và `>=` |
| `nums` đã không tăng | `[3, 2, 1]` vs `[3, 2, 1]` → `[3]` | `L = n` → phải chặn về `n - 1` |
| Không cửa sổ nào là hoán vị | `[1, 2]` vs `[3, 3, 3]` → `[0, 0]` | Đừng quên trả `0` thay vì bỏ qua phần tử |
| Cặp đi lên ngay ngoài mép cửa sổ | `[2, 1]` vs `[2, 1, 3]` → `[2, 0]` | Lỗi lệch chỉ số `last_up` |
| Giá trị lớn `10^5` | `[100000]` | Mảng `diff` phải đủ `max + 1` ô |

Tất cả các ca này đã nằm trong cell test của mục 10.

# 15. Dấu hiệu nhận dạng và bài liên quan

## Dấu hiệu 1 — "So mảng độ dài cố định với **mọi** cửa sổ của một mảng dài"

Nghĩ ngay tới: **sliding window** (cho thông tin dạng đếm/tổng), **rolling hash**, **Z-function / KMP** (cho so khớp thứ tự).

## Dấu hiệu 2 — "Sắp xếp một phần của mảng"

Phần đã sắp xếp có **hình dạng cố định** (không tăng / không giảm). Thay vì sắp xếp thật, hãy hỏi:
"đoạn này đã đơn điệu chưa?" và "có đúng các phần tử không?" — hai câu hỏi rẻ hơn nhiều.

## Dấu hiệu 3 — "Đếm số chỉ số `i` thỏa một điều kiện"

Kiểm tra xem điều kiện có **đơn điệu theo `i`** không (đúng rồi sai mãi, hoặc sai rồi đúng mãi).
Nếu có, tập lời giải là một **đoạn** và đáp án chỉ là `max(0, phải - trái + 1)`.

## Dấu hiệu 4 — "LCP của một mẫu với mọi vị trí"

Z-function trên `pattern + [sep] + text` là công cụ chuẩn; rolling hash + binary search là phương án dự phòng dễ nhớ.

---

## Bài liên quan (cùng kỹ thuật)

| Bài | Kỹ thuật chung với bài này |
|---|---|
| LeetCode 438 — Find All Anagrams in a String | Sliding window + `diff`/`bad` (điều kiện c) |
| LeetCode 567 — Permutation in String | Sliding window kiểm tra hoán vị |
| LeetCode 31 — Next Permutation | Tìm hậu tố không tăng dài nhất (điều kiện b) |
| LeetCode 28 — Find the Index of the First Occurrence in a String | So khớp mẫu: Z-function / KMP / rolling hash |
| LeetCode 2223 — Sum of Scores of Built Strings | Z-function thuần |
| LeetCode 1044 — Longest Duplicate Substring | Rolling hash + binary search |
| LeetCode 3036 — Number of Subarrays That Match a Pattern II | So khớp mẫu trên **mảng số** bằng Z / KMP |

Một hướng khác cho bài này (không trình bày code ở đây): hash **trực tiếp** từng `A_i` — cần "hash của multiset đã sắp xếp",
duy trì bằng segment tree theo giá trị khi thêm dần phần tử từ phải sang trái, O(n log V). Cách đó đúng nhưng nặng hơn và vẫn là xác suất;
quan sát 3 điều kiện cho lời giải gọn và chính xác hơn.

Cell dưới dùng lại đúng các mảnh của bài cho 3 bài liên quan.

In [32]:
# (1) Find All Anagrams (bản cho list số): các vị trí k mà cửa sổ là hoán vị của p
def find_anagram_starts(text, p):
    return [k for k, ok in enumerate(permutation_windows(p, text)) if ok]
print("Anagram starts:", find_anagram_starts([1, 2, 1, 3, 1, 1, 2], [1, 1, 2]))

# (2) strStr bằng Z-function: vị trí đầu tiên mà pattern xuất hiện trong text
def first_occurrence(text, pattern):
    n = len(pattern)
    z = z_function(pattern + [0] + text)
    for k in range(len(text) - n + 1):
        if z[n + 1 + k] == n:
            return k
    return -1
print("First occurrence:", first_occurrence([2, 2, 3, 2, 1, 3, 2, 2], [3, 2, 2]))

# (3) Next Permutation: bước đầu tiên chính là tìm hậu tố không tăng dài nhất
def next_permutation(a):
    a = a[:]
    s = len(a) - 1
    while s > 0 and a[s - 1] >= a[s]:
        s -= 1
    if s == 0:
        return a[::-1]
    p = s - 1                                # phần tử ngay trước hậu tố không tăng
    q = len(a) - 1
    while a[q] <= a[p]:
        q -= 1
    a[p], a[q] = a[q], a[p]
    a[s:] = reversed(a[s:])
    return a
print("Next permutation of [1, 3, 2]:", next_permutation([1, 3, 2]))

Anagram starts: [0, 4]
First occurrence: 5
Next permutation of [1, 3, 2]: [2, 1, 3]


# 16. Bài tập vận dụng

Làm theo thứ tự — độ khó tăng dần. Gợi ý theo 3 mức nằm **sau** phần đề bài tập; đáp án nằm ở **cuối notebook**.

## Bài 1 — Đoán output

Không chạy code, hãy viết ra:

```python
nums = [4, 1, 3, 3]
for i in range(4):
    print(sorted_suffix_array(nums, i))
print(Counter(tuple(sorted_suffix_array(nums, i)) for i in range(4)))
```

Có bao nhiêu **mảng khác nhau**? Mỗi mảng được tạo bởi bao nhiêu chỉ số `i`?

## Bài 2 — Tính bằng tay

Với `nums = [4, 1, 3, 3]`, `target = [4, 3, 3, 1, 4, 1, 3, 3]`, hãy lập bảng:

| k | `W_k` | hoán vị? | `L_k` | `s_k` | `ans[k]` |
|---|---|---|---|---|---|
| 0 | ? | ? | ? | ? | ? |
| ... | | | | | |

Câu hỏi thêm: có bao nhiêu cửa sổ **là hoán vị** của `nums`, và bao nhiêu cửa sổ có đáp án **khác 0**?

## Bài 3 — Điền TODO: đầu hậu tố không tăng dài nhất

Viết `exercise_3_nonincreasing_start(w)` trả về chỉ số nhỏ nhất `s` sao cho `w[s:]` là dãy không tăng.

```python
exercise_3_nonincreasing_start([5, 6, 5])     -> 1
exercise_3_nonincreasing_start([3, 2, 2, 1])  -> 0
exercise_3_nonincreasing_start([1, 2, 3])     -> 2
exercise_3_nonincreasing_start([7])           -> 0
```

In [33]:
def exercise_3_nonincreasing_start(w: List[int]) -> int:
    # TODO 1: bắt đầu từ s = len(w) - 1 (hậu tố 1 phần tử luôn không tăng)
    # TODO 2: khi còn s > 0 và w[s - 1] >= w[s], lùi s sang trái
    # TODO 3: trả về s
    pass


# Tự làm xong rồi mới mở test:
# assert exercise_3_nonincreasing_start([5, 6, 5]) == 1
# assert exercise_3_nonincreasing_start([3, 2, 2, 1]) == 0
# assert exercise_3_nonincreasing_start([1, 2, 3]) == 2
# assert exercise_3_nonincreasing_start([7]) == 0

## Bài 4 — Điền TODO: sliding window kiểm tra hoán vị

Viết `exercise_4_permutation_windows(nums, target)` trả về list `bool`, phần tử thứ `k` là `True` nếu `target[k:k+n]` là hoán vị của `nums`.
Yêu cầu: **O(n + m)** — không gọi `Counter` cho từng cửa sổ.

```python
exercise_4_permutation_windows([5, 6, 5], [5, 6, 5, 5, 6, 5]) -> [True, True, True, True]
exercise_4_permutation_windows([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]) -> [True, True, False, False, False, True]
```

In [34]:
def exercise_4_permutation_windows(nums: List[int], target: List[int]) -> List[bool]:
    # TODO 1: tạo diff (list hoặc defaultdict), cộng +1 cho mỗi x trong nums
    # TODO 2: trừ -1 cho mỗi phần tử của cửa sổ đầu target[0:n]
    # TODO 3: bad = số giá trị có diff khác 0; ghi nhận cửa sổ k = 0
    # TODO 4: với k = 1..m-n: target[k-1] rời (+1), target[k+n-1] vào (-1),
    #         mỗi lần đổi diff[v] thì cập nhật bad theo 3 trường hợp
    raise NotImplementedError


# assert exercise_4_permutation_windows([5, 6, 5], [5, 6, 5, 5, 6, 5]) == [True, True, True, True]
# assert exercise_4_permutation_windows([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]) == [True, True, False, False, False, True]

## Bài 5 — Biến thể: sắp xếp hậu tố theo thứ tự **không giảm**

Giữ nguyên đề, chỉ đổi: hậu tố `nums[i..n-1]` được sắp xếp **không giảm** (tăng dần, cho phép bằng nhau).
Viết `exercise_5_count_ascending_suffix_sorts(nums, target)` chạy O(n + m).

```python
exercise_5_count_ascending_suffix_sorts([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]) -> [3, 0, 0, 0, 0, 0]
```

(Vì sao `ans[0] = 3`? Vì `[2, 2, 3]` vốn đã không giảm: mọi `i` đều cho lại chính `nums`.)

In [35]:
def exercise_5_count_ascending_suffix_sorts(nums: List[int], target: List[int]) -> List[int]:
    # TODO 1: điều kiện (a) và (c) giữ nguyên -> dùng lại lcp_by_z và permutation_windows
    # TODO 2: điều kiện (b) đổi thành "W[i:] không giảm":
    #         thay "vị trí đi lên" bằng "vị trí đi xuống" target[t] > target[t + 1]
    # TODO 3: áp lại công thức max(0, min(L, n - 1) - s + 1) khi là hoán vị
    pass


# assert exercise_5_count_ascending_suffix_sorts([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]) == [3, 0, 0, 0, 0, 0]

## Bài 6 — Mini challenge: sắp xếp **tiền tố** thay vì hậu tố

Đề mới: với mỗi `i` (`0 <= i <= n - 1`), sắp xếp **tiền tố** `nums[0..i]` theo thứ tự không tăng, giữ nguyên `nums[i+1..n-1]`:

```text
B_i = sorted(nums[:i + 1], reverse=True) + nums[i + 1:]
```

`ans[k]` = số chỉ số `i` sao cho `B_i == target[k:k+n]`. Viết `exercise_6_count_valid_prefix_sorts(nums, target)` chạy O(n + m).

```python
exercise_6_count_valid_prefix_sorts([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]) -> [2, 0, 0, 0, 0, 1]
exercise_6_count_valid_prefix_sorts([5, 6, 5], [5, 6, 5, 5, 6, 5])       -> [1, 2, 0, 1]
```

Câu hỏi tư duy trước khi code: 3 điều kiện mới là gì? Tập `i` thỏa từng điều kiện có còn là đoạn liên tục không?

In [36]:
def exercise_6_count_valid_prefix_sorts(nums: List[int], target: List[int]) -> List[int]:
    # TODO 1: viết 3 điều kiện cho W == B_i (hậu tố khớp, đầu không tăng, hoán vị)
    # TODO 2: hậu tố chung dài nhất S_k của W_k và nums: Z-function trên mảng ĐẢO NGƯỢC
    # TODO 3: e_k = chỉ số cuối của tiền tố không tăng dài nhất của W_k (dùng "vị trí đi lên" đầu tiên >= k)
    # TODO 4: đếm i trong đoạn [max(0, n - 1 - S_k), e_k] khi W_k là hoán vị
    pass


# assert exercise_6_count_valid_prefix_sorts([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]) == [2, 0, 0, 0, 0, 1]
# assert exercise_6_count_valid_prefix_sorts([5, 6, 5], [5, 6, 5, 5, 6, 5]) == [1, 2, 0, 1]

## Gợi ý theo 3 mức

Đọc lần lượt từng mức; chỉ đọc mức sau khi mức trước chưa đủ.

| Bài | Mức 1 — hướng tư duy | Mức 2 — cấu trúc / cú pháp | Mức 3 — gần code |
|---|---|---|---|
| 1 | Hậu tố bắt đầu tại `i = 2` và `i = 3` đã không tăng sẵn chưa? | `sorted([1, 3, 3], reverse=True)` là gì? | `A_0 = [4, 3, 3, 1]`; tự tính `A_1`, `A_2`, `A_3` rồi gom. |
| 2 | Tính `eq` trước; chỉ cửa sổ hoán vị mới cần `L`, `s`. | `L` = số phần tử đầu trùng `[4, 1, 3, 3]`; `s` = đầu hậu tố không tăng dài nhất. | `W_4 = [4, 1, 3, 3] = nums` → `L = 4`, cắt về `3`; `s = 2`. |
| 3 | Hậu tố 1 phần tử luôn không tăng; mở rộng sang trái khi còn được. | Vòng `while` với điều kiện kép `s > 0 and ...`. | `while s > 0 and w[s - 1] >= w[s]: s -= 1` |
| 4 | Chỉ 2 phần tử thay đổi khi trượt. | `diff` + `bad`, ba trường hợp cân/lệch. | `before = diff[v]; diff[v] += d; if before == 0: bad += 1 elif diff[v] == 0: bad -= 1` |
| 5 | Chỉ điều kiện (b) đổi chiều. | `last_down[t]` = vị trí `t' <= t` lớn nhất có `target[t'] > target[t'+1]`. | `j = last_down[k + n - 2]; s = j - k + 1 if j >= k else 0` |
| 6 | Đảo vai trò: đuôi giữ nguyên, đầu được sắp xếp. | Hậu tố chung = tiền tố chung của hai mảng đảo ngược. Vị trí của `target[k + n - 1]` trong `target[::-1]` là `m - n - k`. | `S = z[n + 1 + (m - n - k)]`; `lo = max(0, n - 1 - S)`; `e = next_up[k] - k` nếu `next_up[k] <= k + n - 2`, ngược lại `n - 1`. |

# Đáp án tham khảo

Bạn nên tự làm trước khi mở phần này.

---

## Đáp án Bài 1

```text
[4, 3, 3, 1]     <- i = 0
[4, 3, 3, 1]     <- i = 1: [4] + sorted([1, 3, 3], giảm dần)
[4, 1, 3, 3]     <- i = 2: hậu tố [3, 3] đã không tăng
[4, 1, 3, 3]     <- i = 3: A_{n-1} luôn bằng nums
Counter({(4, 3, 3, 1): 2, (4, 1, 3, 3): 2})
```

Có **2** mảng khác nhau, mỗi mảng được tạo bởi **2** chỉ số.

## Đáp án Bài 2

| k | `W_k` | hoán vị? | `L_k` | `s_k` | `ans[k]` |
|---|---|---|---|---|---|
| 0 | `[4, 3, 3, 1]` | có | 1 | 0 | 2 (`i ∈ [0, 1]`) |
| 1 | `[3, 3, 1, 4]` | có | 0 | 3 | 0 (đoạn `[3, 0]` rỗng) |
| 2 | `[3, 1, 4, 1]` | không (hai số 1, thiếu một số 3) | (0) | (2) | 0 |
| 3 | `[1, 4, 1, 3]` | không | (0) | (3) | 0 |
| 4 | `[4, 1, 3, 3]` | có | 4 | 2 | 2 (`i ∈ [2, 3]`) |

(Giá trị trong ngoặc: tính được nhưng không cần dùng, vì cửa sổ đã bị loại ở điều kiện hoán vị.)

Có **3** cửa sổ là hoán vị của `nums` (`k = 0, 1, 4`), nhưng chỉ **2** cửa sổ có đáp án khác 0 → `ans = [2, 0, 0, 0, 2]`.
Cửa sổ `k = 1` là minh họa cho "hoán vị là cần nhưng chưa đủ". Cẩn thận khi cắt cửa sổ bằng tay: `W_2 = target[2:6] = [3, 1, 4, 1]`.

In [37]:
nums = [4, 1, 3, 3]
for i in range(4):
    print(sorted_suffix_array(nums, i))
print(Counter(tuple(sorted_suffix_array(nums, i)) for i in range(4)))

target = [4, 3, 3, 1, 4, 1, 3, 3]
eq = permutation_windows(nums, target)
L = lcp_by_z(nums, target)
s = nonincreasing_starts(target, len(nums))
ans = count_valid_suffix_sorts_z(nums, target)
for k in range(len(eq)):
    print(f"k={k}: W={target[k:k + 4]}, hoán vị={eq[k]}, L={L[k]}, s={s[k]}, ans={ans[k]}")
assert ans == [2, 0, 0, 0, 2] == count_valid_suffix_sorts_bruteforce(nums, target)
assert eq == [True, True, False, False, True] and L == [1, 0, 0, 0, 4] and s == [0, 3, 2, 3, 2]

[4, 3, 3, 1]
[4, 3, 3, 1]
[4, 1, 3, 3]
[4, 1, 3, 3]
Counter({(4, 3, 3, 1): 2, (4, 1, 3, 3): 2})
k=0: W=[4, 3, 3, 1], hoán vị=True, L=1, s=0, ans=2
k=1: W=[3, 3, 1, 4], hoán vị=True, L=0, s=3, ans=0
k=2: W=[3, 1, 4, 1], hoán vị=False, L=0, s=2, ans=0
k=3: W=[1, 4, 1, 3], hoán vị=False, L=0, s=3, ans=0
k=4: W=[4, 1, 3, 3], hoán vị=True, L=4, s=2, ans=2


## Đáp án Bài 3

Hậu tố 1 phần tử luôn không tăng, nên bắt đầu từ `s = len(w) - 1` và lùi sang trái khi cặp `(s-1, s)` vẫn "không tăng" (`>=`).

In [38]:
def answer_3_nonincreasing_start(w: List[int]) -> int:
    s = len(w) - 1
    while s > 0 and w[s - 1] >= w[s]:
        s -= 1
    return s

assert answer_3_nonincreasing_start([5, 6, 5]) == 1
assert answer_3_nonincreasing_start([3, 2, 2, 1]) == 0
assert answer_3_nonincreasing_start([1, 2, 3]) == 2
assert answer_3_nonincreasing_start([7]) == 0

rng = random.Random(3)
for _ in range(300):
    w = [rng.randint(1, 3) for _ in range(rng.randint(1, 7))]
    expected = min(i for i in range(len(w)) if is_nonincreasing(w[i:]))
    assert answer_3_nonincreasing_start(w) == expected
print("Đáp án Bài 3: PASS")

Đáp án Bài 3: PASS


## Đáp án Bài 4

Dùng `defaultdict(int)` để không phải biết trước giá trị lớn nhất; so với `Counter` từng cửa sổ trên dữ liệu ngẫu nhiên.

In [39]:
def answer_4_permutation_windows(nums: List[int], target: List[int]) -> List[bool]:
    n, m = len(nums), len(target)
    diff = defaultdict(int)
    for x in nums:
        diff[x] += 1
    for j in range(n):
        diff[target[j]] -= 1
    bad = sum(1 for d in diff.values() if d != 0)
    result = [bad == 0]
    for k in range(1, m - n + 1):
        for v, d in ((target[k - 1], +1), (target[k + n - 1], -1)):
            before = diff[v]
            diff[v] += d
            if before == 0:
                bad += 1
            elif diff[v] == 0:
                bad -= 1
        result.append(bad == 0)
    return result

assert answer_4_permutation_windows([5, 6, 5], [5, 6, 5, 5, 6, 5]) == [True, True, True, True]
assert answer_4_permutation_windows([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]) == [True, True, False, False, False, True]
rng = random.Random(4)
for _ in range(300):
    n = rng.randint(1, 5)
    nums = [rng.randint(1, 3) for _ in range(n)]
    target = [rng.randint(1, 3) for _ in range(rng.randint(n, n + 6))]
    expected = [Counter(target[k:k + n]) == Counter(nums) for k in range(len(target) - n + 1)]
    assert answer_4_permutation_windows(nums, target) == expected
print("Đáp án Bài 4: PASS")

Đáp án Bài 4: PASS


## Đáp án Bài 5

Chỉ điều kiện (b) đổi chiều: "đuôi không giảm" ⇔ không có **vị trí đi xuống** (`target[t] > target[t + 1]`) trong đuôi.
Ta viết thêm một brute force cho biến thể để kiểm chứng.

In [40]:
def answer_5_count_ascending_suffix_sorts(nums: List[int], target: List[int]) -> List[int]:
    n, m = len(nums), len(target)
    eq = permutation_windows(nums, target)
    L = lcp_by_z(nums, target)
    last_down = [-1] * m
    prev = -1
    for t in range(m - 1):
        if target[t] > target[t + 1]:
            prev = t
        last_down[t] = prev
    ans = []
    for k in range(m - n + 1):
        j = last_down[k + n - 2] if n >= 2 else -1
        s = j - k + 1 if j >= k else 0
        ans.append(max(0, min(L[k], n - 1) - s + 1) if eq[k] else 0)
    return ans

def bruteforce_5(nums, target):
    n = len(nums)
    arrays = [nums[:i] + sorted(nums[i:]) for i in range(n)]
    return [sum(a == target[k:k + n] for a in arrays) for k in range(len(target) - n + 1)]

assert answer_5_count_ascending_suffix_sorts([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]) == [3, 0, 0, 0, 0, 0]
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 6)
    nums = [rng.randint(1, 3) for _ in range(n)]
    target = [rng.randint(1, 3) for _ in range(rng.randint(n, n + 7))]
    if rng.random() < 0.5:
        i, p = rng.randrange(n), rng.randint(0, len(target) - n)
        target[p:p + n] = nums[:i] + sorted(nums[i:])
    assert answer_5_count_ascending_suffix_sorts(nums, target) == bruteforce_5(nums, target)
print("Đáp án Bài 5: PASS")

Đáp án Bài 5: PASS


## Đáp án Bài 6

Ba điều kiện cho `W == B_i`:

- **(a') Hậu tố khớp:** `W[i+1:] == nums[i+1:]`. Gọi `S_k` = độ dài hậu tố chung dài nhất của `W_k` và `nums`; (a') ⇔ `n - 1 - i <= S_k` ⇔ `i >= n - 1 - S_k`.
- **(b') Đầu không tăng:** `W[0..i]` không tăng ⇔ `i <= e_k`, với `e_k` = chỉ số cuối của tiền tố không tăng dài nhất.
- **(c) Hoán vị:** như cũ.

Đáp án: `max(0, e_k - max(0, n - 1 - S_k) + 1)` nếu hoán vị. (`max(0, ...)` ở cận trái đóng vai trò giống `min(L, n - 1)` ở bài gốc.)

`S_k` lấy từ Z-function trên `nums[::-1] + [0] + target[::-1]`; `e_k` lấy từ mảng `next_up` (vị trí đi lên **đầu tiên** `>= t`, tính từ phải sang trái).

In [41]:
def answer_6_count_valid_prefix_sorts(nums: List[int], target: List[int]) -> List[int]:
    n, m = len(nums), len(target)
    eq = permutation_windows(nums, target)
    z = z_function(nums[::-1] + [0] + target[::-1])
    NONE = m                                   # "không có vị trí đi lên"
    next_up = [NONE] * m
    nxt = NONE
    for t in range(m - 2, -1, -1):
        if target[t] < target[t + 1]:
            nxt = t
        next_up[t] = nxt
    ans = []
    for k in range(m - n + 1):
        if not eq[k]:
            ans.append(0)
            continue
        S = z[n + 1 + (m - n - k)]             # hậu tố chung của W_k và nums
        lo = max(0, n - 1 - S)
        j = next_up[k] if n >= 2 else NONE
        e = j - k if j <= k + n - 2 else n - 1
        ans.append(max(0, e - lo + 1))
    return ans

def bruteforce_6(nums, target):
    n = len(nums)
    arrays = [sorted(nums[:i + 1], reverse=True) + nums[i + 1:] for i in range(n)]
    return [sum(a == target[k:k + n] for a in arrays) for k in range(len(target) - n + 1)]

assert answer_6_count_valid_prefix_sorts([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]) == [2, 0, 0, 0, 0, 1]
assert answer_6_count_valid_prefix_sorts([5, 6, 5], [5, 6, 5, 5, 6, 5]) == [1, 2, 0, 1]
rng = random.Random(6)
for _ in range(300):
    n = rng.randint(1, 6)
    nums = [rng.randint(1, 3) for _ in range(n)]
    target = [rng.randint(1, 3) for _ in range(rng.randint(n, n + 7))]
    if rng.random() < 0.5:
        i, p = rng.randrange(n), rng.randint(0, len(target) - n)
        target[p:p + n] = sorted(nums[:i + 1], reverse=True) + nums[i + 1:]
    assert answer_6_count_valid_prefix_sorts(nums, target) == bruteforce_6(nums, target)
print("Đáp án Bài 6: PASS")

Đáp án Bài 6: PASS


# 17. Tổng kết

## Ý tưởng cốt lõi trong một câu

> `A_i` = "tiền tố giữ nguyên" + "đuôi không tăng". Nên `W_k == A_i` ⇔ **tiền tố khớp** (`i <= L_k`) + **đuôi không tăng** (`i >= s_k`) + **là hoán vị** — và số `i` hợp lệ là độ dài một đoạn.

## Template dễ nhớ

```python
# với mỗi cửa sổ W_k = target[k:k+n]:
eq_k = (bad == 0)                     # sliding window diff/bad
L_k  = z[n + 1 + k]                   # Z-function trên nums + [0] + target
j    = last_up[k + n - 2]             # vị trí đi lên cuối cùng trong cửa sổ (n >= 2)
s_k  = j - k + 1 if j >= k else 0
ans[k] = max(0, min(L_k, n - 1) - s_k + 1) if eq_k else 0
```

## Hành trình tư duy

1. **Brute force** theo đúng chữ của đề → trọng tài.
2. **Bỏ việc lặp lại**: dựng `A_i` một lần, gom bằng `Counter`.
3. **Hiểu hình dạng** của đối tượng (`A_i`) → tách thành điều kiện nhỏ, mỗi điều kiện **đơn điệu theo `i`**.
4. **Mỗi điều kiện một công cụ kinh điển**: sliding window, mảng tiền xử lý, rolling hash / Z-function.

Bài học chuyển giao: khi đề "sắp xếp rồi so sánh", hãy hỏi *"kết quả sắp xếp trông như thế nào?"* thay vì sắp xếp thật.

# Checklist trước khi submit

- [ ] Đọc kỹ: đếm **số chỉ số `i`**, không phải số mảng khác nhau.
- [ ] "Không tăng" dùng `>=` (cho phép bằng nhau), "đi lên" là `target[t] < target[t + 1]`.
- [ ] Có kiểm tra **hoán vị** (`bad == 0`) trước khi dùng `L_k`, `s_k`.
- [ ] Chặn `L_k` bằng `min(L_k, n - 1)` vì `i <= n - 1`.
- [ ] Tra `last_up[k + n - 2]` (cặp kề cuối cùng **trong** cửa sổ), và xử lý riêng `n = 1`.
- [ ] Separator khi ghép Z-function là giá trị **không xuất hiện** trong input (ở đây `0`).
- [ ] Nếu dùng hash: `BASE > max giá trị`, `MOD` lớn (ví dụ `2^61 - 1`), cân nhắc `BASE` ngẫu nhiên.
- [ ] Mảng `diff` đủ `max(max(nums), max(target)) + 1` ô.
- [ ] Độ dài kết quả đúng `m - n + 1`; test `n = 1`, `n = m`, toàn số giống nhau.
- [ ] Code nộp không `print`, đúng signature trong `class Solution`, có `from typing import List`.